# Solaris AI Version 8 — standalone Kaggle notebook

Run all cells in order in a fresh Kaggle session with Internet enabled. Only tabular observations are downloaded. SRS is optional and may be excluded after coverage checks. All output is written under `/kaggle/working/solaris_ai_v8/`. Version 6 and Version 7 files are untouched.

The SuryaBench label is a full-disk 24-hour M/X outcome. The previously examined 2020–2024 outcomes are reserved for descriptive comparison and cannot establish independent performance. Model choices use earlier years only. SHARP is admitted only after numeric measurements and historical coverage are verified; SRS availability is conservatively delayed by 24 hours. Flare-catalog availability delay is an approximation, not a verified historical publication time.

GOES measurement features are delayed an additional hour, SHARP by 12 hours, SRS by 24 hours and the retrospective flare catalog by 24 hours after peak. These are conservative modeling assumptions that require operational verification. A trailing 27-day GOES proxy represents recent solar activity without using future solar-cycle classifications.


In [ ]:
# Kaggle setup. Numeric data only.
import os
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "20")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "60")
from pathlib import Path
Path("/kaggle/working/solaris_ai_v8").mkdir(parents=True, exist_ok=True)
%pip install -q optuna catboost shap drms astropy pyarrow xgboost lightgbm huggingface_hub xarray requests netCDF4


In [ ]:

from huggingface_hub import snapshot_download
import os

dataset_path = snapshot_download(
    repo_id="nasa-ibm-ai4science/surya-bench-flare-forecasting",
    repo_type="dataset",
    local_dir="/kaggle/working/solaris_ai_v8/surya_dataset",
    allow_patterns=["*.csv"]
)

print("Dataset downloaded successfully!\n")

for file in os.listdir(dataset_path):
    print(file)


In [ ]:
# Download annual NOAA science-quality GOES XRS history and create leakage-safe hourly features.
from pathlib import Path
import requests, xarray as xr, numpy as np, pandas as pd, time, os
noaa_dir = Path('/kaggle/working/solaris_ai_v8/noaa_goes_annual')
noaa_dir.mkdir(parents=True, exist_ok=True)
annual_urls = {}
for year in range(2010, 2025):
    if year == 2010:
        sat, root, version = 'g14', 'https://www.ncei.noaa.gov/data/goes-space-environment-monitor/access/science/xrs/goes14/xrsf-l2-avg1m_science/', 'v2-2-1'
    elif 2011 <= year <= 2016:
        sat, root, version = 'g15', 'https://www.ncei.noaa.gov/data/goes-space-environment-monitor/access/science/xrs/goes15/xrsf-l2-avg1m_science/', 'v2-2-1'
    else:
        sat, root, version = 'g16', 'https://data.ngdc.noaa.gov/platforms/solar-space-observing-satellites/goes/goes16/l2/data/xrsf-l2-avg1m_science/', 'v2-2-2'
    name = f'sci_xrsf-l2-avg1m_{sat}_y{year}_{version}.nc'
    annual_urls[year] = root + name

local_nc = {}
for year, url in annual_urls.items():
    dest = noaa_dir / url.rsplit('/', 1)[-1]
    if not dest.exists() or dest.stat().st_size < 10000:
        part = dest.with_suffix(dest.suffix + '.part')
        print(f'Downloading {year} {dest.name}', flush=True)
        for attempt in range(2):
            try:
                deadline=time.monotonic()+180
                with requests.get(url, stream=True, timeout=(8, 20)) as response:
                    response.raise_for_status()
                    with open(part, 'wb') as f:
                        for chunk in response.iter_content(1024 * 1024):
                            if time.monotonic()>deadline: raise TimeoutError('GOES file exceeded 180 second budget')
                            if chunk: f.write(chunk)
                if part.stat().st_size < 10000: raise ValueError('GOES file is unexpectedly small')
                part.replace(dest)
                break
            except (requests.RequestException, OSError, TimeoutError, ValueError):
                if part.exists(): part.unlink()
                if attempt == 1: raise
                time.sleep(1)
    local_nc[year] = dest
    print(f'{year}: {dest.stat().st_size/1024/1024:.1f} MiB', flush=True)

# Remove samples from bad, eclipsed, interpolated, non-finite, or fill-valued records.
hourly_parts = []
quality_report = []
for year, path in local_nc.items():
    with xr.open_dataset(path, decode_cf=True, mask_and_scale=True) as ds:
        times = pd.to_datetime(ds['time'].values, utc=True)
        minute = pd.DataFrame(index=times)
        for band in ('xrsa', 'xrsb'):
            flux = np.asarray(ds[f'{band}_flux'].values, dtype='float64').reshape(-1)
            flag_name = f'{band}_flag'
            flags = np.asarray(ds[flag_name].values).reshape(-1) if flag_name in ds else np.zeros(len(flux))
            fill = ds[f'{band}_flux'].attrs.get('_FillValue', -9999)
            valid = np.isfinite(flux) & (flux > 0) & (flux != fill) & np.isfinite(flags) & (flags == 0)
            minute[f'{band}_log'] = np.where(valid, np.log10(np.maximum(flux, 1e-12)), np.nan)
            minute[f'{band}_ok'] = valid.astype('float32')
            quality_report.append((year, band, len(flux), int(valid.sum()), int((~np.isfinite(flux) | (flux == fill)).sum()), int((np.isfinite(flags) & (flags != 0)).sum())))
        minute.index.name = 'timestamp'
        # Each hourly bin is [t-1 hour, t), so no measurements from the label's future window enter features at t.
        h = minute.resample('1h', closed='left', label='right').agg({
            'xrsa_log':['mean','max','std'], 'xrsa_ok':'mean',
            'xrsb_log':['mean','max','std'], 'xrsb_ok':'mean'
        })
        h.columns = [f'{a}_{b}' for a,b in h.columns]
        h = h.rename(columns={'xrsa_ok_mean':'xrsa_coverage','xrsb_ok_mean':'xrsb_coverage'})
        hourly_parts.append(h)
        print(f'Parsed {year}: {len(minute):,} minute rows, {len(h):,} hourly bins', flush=True)

goesh = pd.concat(hourly_parts).sort_index()
goesh = goesh[~goesh.index.duplicated(keep='last')]
goesh = goesh.asfreq('1h')  # Preserve clock-hour meaning across data gaps.
# Causal trailing history summaries across the preceding 1, 6, 24, and 72 completed hourly bins.
base_cols = ['xrsa_log_mean','xrsa_log_max','xrsb_log_mean','xrsb_log_max']
for window in (6, 24, 72):
    roll = goesh[base_cols].rolling(window=window, min_periods=max(1, window//2))
    for col in base_cols:
        goesh[f'{col}_roll{window}h_mean'] = roll[col].mean()
        goesh[f'{col}_roll{window}h_max'] = roll[col].max()
for col in ('xrsa_coverage','xrsb_coverage'):
    goesh[f'{col}_roll24h_mean'] = goesh[col].rolling(24, min_periods=12).mean()
# Calendar cycles only (no year index or target-derived fields).
idx = goesh.index
hour_of_day = idx.hour + idx.minute/60
annual_phase = (idx.dayofyear - 1) / 365.25 * 2*np.pi
goesh['hour_sin'] = np.sin(2*np.pi*hour_of_day/24)
goesh['hour_cos'] = np.cos(2*np.pi*hour_of_day/24)
goesh['annual_sin'] = np.sin(annual_phase)
goesh['annual_cos'] = np.cos(annual_phase)
feature_cols = list(goesh.columns)
print('NOAA downloads:', len(local_nc), 'files;', round(sum(p.stat().st_size for p in local_nc.values())/1024/1024,1), 'MiB total')
print('Hourly feature table:', goesh.shape, goesh.index.min(), 'to', goesh.index.max())
print('Features:', feature_cols)
print('Quality counts (year, band, total, valid, missing/fill, nonzero flags), first/last examples:')
print(quality_report[:4], '...', quality_report[-4:])

In [ ]:
# Version 7 inputs: NOAA GOES event history plus Version 6 GOES features and labels.
from pathlib import Path
import os, re, json, time, warnings, requests
import numpy as np
import pandas as pd

V7_ROOT = Path("/kaggle/working/solaris_ai_v8")
V7_ROOT.mkdir(parents=True, exist_ok=True)
V7_CACHE = V7_ROOT / "cache"
V7_CACHE.mkdir(exist_ok=True)
V7_START = pd.Timestamp("2010-01-01", tz="UTC")
V7_END_EXCLUSIVE = pd.Timestamp("2025-01-01", tz="UTC")
V7_SHARP_DELAY_HOURS = 12
V7_EVENT_DELAY_HOURS = 24
V7_RANDOM_SEED = 42
V7_SOURCE_LIMITATIONS = []

assert "goesh" in globals() and "feature_cols" in globals(), "Run Version 6 GOES feature cell first."
assert len(feature_cols) == 38, f"Expected Version 6's 38 GOES features; found {len(feature_cols)}."
assert all(pd.api.types.is_numeric_dtype(goesh[c]) for c in feature_cols)
v7_goes = goesh.loc[(goesh.index >= V7_START) & (goesh.index < V7_END_EXCLUSIVE), feature_cols].copy()
# Allow one additional hour for GOES data availability; deterministic calendar features need no delay.
v8_calendar_cols={"hour_sin","hour_cos","annual_sin","annual_cos"}
v8_measurement_cols=[c for c in feature_cols if c not in v8_calendar_cols]
v7_goes.loc[:,v8_measurement_cols]=v7_goes[v8_measurement_cols].shift(1)
v7_goes = v7_goes.replace([np.inf, -np.inf], np.nan)
assert v7_goes.index.is_monotonic_increasing and v7_goes.index.is_unique and v7_goes.index.tz is not None
v8_goes_coverage=v7_goes.groupby(v7_goes.index.year).agg(rows=(feature_cols[0],'size'),first_feature_observed=(feature_cols[0],'count'))
v8_goes_coverage['first_feature_fraction']=v8_goes_coverage['first_feature_observed']/v8_goes_coverage['rows']
v8_goes_coverage.to_csv(V7_ROOT/'goes_annual_coverage.csv')
print('GOES annual coverage:'); print(v8_goes_coverage.to_string())
v7_label_csv = Path("/kaggle/working/solaris_ai_v8/surya_dataset/data.csv")
# Read timestamps first; target values are loaded only for development rows before 2020.
v7_label_time_frame = pd.read_csv(v7_label_csv, usecols=["timestamp"])
v7_label_row_times = pd.to_datetime(v7_label_time_frame["timestamp"], utc=True, errors="coerce")
assert v7_label_row_times.notna().all(), "SuryaBench contains an unparseable timestamp."
v7_pretest_limit = pd.Timestamp("2020-01-01", tz="UTC")
v7_pretest_row_ids = np.flatnonzero(v7_label_row_times < v7_pretest_limit)
v7_pretest_row_id_set = set(map(int, v7_pretest_row_ids))
v7_pretest_label_frame = pd.read_csv(
    v7_label_csv, usecols=["timestamp", "label_max"],
    skiprows=lambda row: row > 0 and (row - 1) not in v7_pretest_row_id_set,
    parse_dates=["timestamp"],
)
assert len(v7_pretest_label_frame) == len(v7_pretest_row_ids), "Pre-2020 label row selection did not match the timestamp index."
v7_pretest_label_frame["timestamp"] = pd.to_datetime(v7_pretest_label_frame["timestamp"], utc=True)
v7_labels = v7_pretest_label_frame[["timestamp", "label_max"]].drop_duplicates("timestamp").set_index("timestamp").sort_index()
v7_labels = v7_labels.loc[(v7_labels.index >= V7_START) & (v7_labels.index < v7_pretest_limit)]
v7_pretest_labels = v7_labels.loc[v7_labels.index < pd.Timestamp("2018-01-01", tz="UTC"), "label_max"]
assert v7_pretest_labels.dropna().nunique() == 2
print("GOES:", v7_goes.shape, "| pre-2020 development labels:", v7_labels.shape)
print("Evaluation outcome values for 2020–2024 remain unread until the final evaluation cell.")

# Official mission-length NOAA composite; one numerical row per flare, no images.
v7_flare_url = "https://data.ngdc.noaa.gov/platforms/solar-space-observing-satellites/goes/multi/l2/data/xrsf-l2-flrpt_science/csv/"
v7_flare_csv = V7_CACHE / "noaa_goes_flare_report.csv"
v7_flare_source_ok = False
try:
    if not v7_flare_csv.exists():
        v7_html = requests.get(v7_flare_url, timeout=(8, 20))
        v7_html.raise_for_status()
        v7_matches = re.findall(r"(?:href=[\"']?)([^\"' >]*sci_xrsf-l2-flrpt_geo_s\d+_e\d+_v[\d-]+\.csv)", v7_html.text, flags=re.I)
        if not v7_matches:
            v7_matches = re.findall(r"sci_xrsf-l2-flrpt_geo_s\d+_e\d+_v[\d-]+\.csv", v7_html.text, flags=re.I)
        if not v7_matches:
            raise RuntimeError("NOAA Flare Report index did not expose its composite CSV.")
        v7_name = v7_matches[0].split("/")[-1]
        v8_flare_deadline=time.monotonic()+180
        with requests.get(v7_flare_url + v7_name, stream=True, timeout=(8, 20)) as v7_response:
            v7_response.raise_for_status()
            with open(v7_flare_csv.with_suffix(".part"), "wb") as v7_out:
                for v7_chunk in v7_response.iter_content(1024 * 1024):
                    if time.monotonic()>v8_flare_deadline:
                        raise TimeoutError("NOAA flare catalog exceeded 180-second download budget")
                    if v7_chunk:
                        v7_out.write(v7_chunk)
        v7_flare_csv.with_suffix(".part").replace(v7_flare_csv)
    v7_flare = pd.read_csv(v7_flare_csv, low_memory=False)
    assert {"time", "flare_class"}.issubset(v7_flare.columns), f"Unexpected NOAA Flare Report schema: {v7_flare.columns.tolist()}"
    v7_flare["peak_time"] = pd.to_datetime(v7_flare["time"], utc=True, errors="coerce")
    v7_flare["class_letter"] = v7_flare["flare_class"].astype(str).str.extract(r"^\s*([A-Z])", expand=False)
    v7_flare = v7_flare.loc[v7_flare["class_letter"].isin(["M", "X"]) & v7_flare["peak_time"].notna()].copy()
    v7_flare["available_at"] = v7_flare["peak_time"] + pd.Timedelta(hours=V7_EVENT_DELAY_HOURS)
    v7_flare["available_hour"] = v7_flare["available_at"].dt.ceil("h")
    v7_flare = v7_flare.drop_duplicates(["peak_time", "flare_class"]).sort_values("available_hour")
    if len(v7_flare)<100 or (v7_flare['peak_time'].max()-v7_flare['peak_time'].min())<pd.Timedelta(days=365):
        raise ValueError('NOAA event catalog lacks sufficient historical M/X event coverage')
    v8_event_annual=v7_flare.groupby(v7_flare['peak_time'].dt.year).size()
    v8_event_annual.to_csv(V7_ROOT/'flare_event_annual_coverage.csv',header=['events'])
    print('NOAA catalog event years:',int(v8_event_annual.index.min()),'to',int(v8_event_annual.index.max()))
    v7_flare_source_ok = True
except Exception as v7_error:
    V7_SOURCE_LIMITATIONS.append(f"NOAA Flare Report unavailable: {type(v7_error).__name__}: {v7_error}")
    v7_flare = pd.DataFrame(columns=["peak_time", "class_letter", "available_at", "available_hour"])
    print("LIMITATION:", V7_SOURCE_LIMITATIONS[-1])
print("NOAA flare report M/X events:", len(v7_flare), "| source available:", v7_flare_source_ok, "| availability delay:", V7_EVENT_DELAY_HOURS, "h")

In [ ]:
# Reuse a valid Version 6 Phase 2 model; rebuild only its compact RF reference if no usable artifact exists.
from pathlib import Path
import joblib
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix

v7_v6_candidates = [
    Path("/kaggle/working/solaris_ai_random_forest_phase2.joblib"),
    Path("/kaggle/working/solaris_ai_random_forest_optimized.joblib"),
]
v7_v6_bundle, v7_v6_artifact_used, v7_v6_load_errors = None, None, []
for v7_candidate_path in v7_v6_candidates:
    if not v7_candidate_path.is_file():
        continue
    try:
        v7_candidate = joblib.load(v7_candidate_path)
        v7_required = {"model", "probability_calibrator", "feature_columns", "operating_threshold"}
        if not v7_required.issubset(v7_candidate):
            raise ValueError(f"missing bundle fields: {sorted(v7_required-set(v7_candidate))}")
        if list(v7_candidate["feature_columns"]) != list(feature_cols):
            raise ValueError("artifact feature order differs from the preserved Version 6 GOES schema")
        v7_probe = goesh.loc[goesh.index < pd.Timestamp("2018-01-01", tz="UTC"), feature_cols].iloc[:4]
        if len(v7_probe) == 0 or v7_candidate["model"].predict_proba(v7_probe).shape != (len(v7_probe), 2):
            raise ValueError("artifact did not produce two-class probabilities on Version 6 features")
        v7_v6_bundle, v7_v6_artifact_used = v7_candidate, str(v7_candidate_path)
        print("Reusing verified Version 6 Phase 2 artifact:", v7_v6_artifact_used)
        break
    except Exception as v7_error:
        v7_v6_load_errors.append(f"{v7_candidate_path}: {type(v7_error).__name__}: {v7_error}")
        print("Version 6 artifact unusable; checking next path:", v7_v6_load_errors[-1])

if v7_v6_bundle is None and all(k in globals() for k in ["rfopt_pipeline","rfopt_calibrator","rfopt_feature_cols","rfopt_operating_threshold"]):
    if list(rfopt_feature_cols) != list(feature_cols):
        raise ValueError("In-memory Version 6 Phase 2 feature order differs from the preserved GOES schema.")
    v7_v6_bundle={"model":rfopt_pipeline,"preprocessing_pipeline":rfopt_pipeline,"probability_calibrator":rfopt_calibrator,"feature_columns":list(rfopt_feature_cols),"operating_threshold":float(rfopt_operating_threshold)}
    v7_v6_copy_path=V7_ROOT/"v6_phase2_reference_reused.joblib"
    joblib.dump(v7_v6_bundle,v7_v6_copy_path,compress=3)
    v7_v6_bundle=joblib.load(v7_v6_copy_path)
    v7_v6_artifact_used=str(v7_v6_copy_path)
    print("Reused and copied the verified in-memory Version 6 Phase 2 artifact:",v7_v6_artifact_used)

if v7_v6_bundle is None:
    print("No usable saved Phase 2 artifact; regenerating the compact Version 6 RF reference from pre-test data only.")
    v7_v6_md = v7_labels[["label_max"]].join(goesh[feature_cols], how="inner").sort_index()
    v7_v6_md = v7_v6_md.loc[v7_v6_md.index < pd.Timestamp("2018-01-01", tz="UTC")]
    v7_v6_y = v7_v6_md["label_max"].astype("int8")
    v7_v6_X = v7_v6_md[feature_cols].replace([np.inf, -np.inf], np.nan)
    v7_2015 = pd.Timestamp("2015-01-01", tz="UTC")
    v7_2016 = pd.Timestamp("2016-01-01", tz="UTC")
    v7_2017 = pd.Timestamp("2017-01-01", tz="UTC")
    v7_2018 = pd.Timestamp("2018-01-01", tz="UTC")
    v7_purge = pd.Timedelta(hours=24)
    v7_rf_tune_fit = v7_v6_md.index < v7_2015-v7_purge
    v7_rf_tune_val = (v7_v6_md.index >= v7_2015) & (v7_v6_md.index < v7_2016-v7_purge)
    v7_rf_fit = v7_v6_md.index < v7_2016-v7_purge
    v7_rf_cal = (v7_v6_md.index >= v7_2016) & (v7_v6_md.index < v7_2017-v7_purge)
    v7_rf_thr = (v7_v6_md.index >= v7_2017) & (v7_v6_md.index < v7_2018-v7_purge)
    for v7_name, v7_mask in [("2010-2014 fit",v7_rf_tune_fit),("2015 tuning",v7_rf_tune_val),("final fit",v7_rf_fit),("2016 calibration",v7_rf_cal),("2017 threshold",v7_rf_thr)]:
        if int(v7_mask.sum()) == 0 or v7_v6_y.loc[v7_mask].nunique() < 2:
            raise ValueError(f"Cannot regenerate Version 6 Phase 2 reference: invalid {v7_name} split.")
    v7_rf_grid = [
        {"n_estimators":350,"max_depth":12,"min_samples_leaf":20,"max_features":"sqrt"},
        {"n_estimators":200,"max_depth":12,"min_samples_leaf":10,"max_features":"sqrt"},
        {"n_estimators":200,"max_depth":16,"min_samples_leaf":5,"max_features":"sqrt"},
        {"n_estimators":200,"max_depth":20,"min_samples_leaf":10,"max_features":0.7},
    ]
    v7_rf_rows, v7_rf_candidates = [], []
    for v7_i, v7_params in enumerate(v7_rf_grid, 1):
        v7_model = Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
            ("model", RandomForestClassifier(**v7_params, class_weight="balanced_subsample", n_jobs=4, random_state=42)),
        ])
        v7_model.fit(v7_v6_X.loc[v7_rf_tune_fit], v7_v6_y.loc[v7_rf_tune_fit])
        v7_p = v7_model.predict_proba(v7_v6_X.loc[v7_rf_tune_val])[:, 1]
        v7_ap = average_precision_score(v7_v6_y.loc[v7_rf_tune_val], v7_p)
        v7_auc = roc_auc_score(v7_v6_y.loc[v7_rf_tune_val], v7_p)
        v7_score = 0.70*v7_ap + 0.30*v7_auc
        v7_rf_rows.append({"candidate":v7_i,"PR_AUC":v7_ap,"ROC_AUC":v7_auc,"Brier_raw":brier_score_loss(v7_v6_y.loc[v7_rf_tune_val],v7_p),"selection_score":v7_score,**v7_params})
        v7_rf_candidates.append((v7_score, v7_params))
        print(f"V6 RF recovery candidate {v7_i}/{len(v7_rf_grid)}: AP={v7_ap:.4f}, ROC-AUC={v7_auc:.4f}", flush=True)
    v7_rf_tuning = pd.DataFrame(v7_rf_rows).sort_values(["selection_score","ROC_AUC"], ascending=False).reset_index(drop=True)
    v7_rf_best = dict(next(params for score,params in v7_rf_candidates if np.isclose(score,v7_rf_tuning.iloc[0]["selection_score"])))
    v7_rf_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
        ("model", RandomForestClassifier(**v7_rf_best, class_weight="balanced_subsample", n_jobs=4, random_state=42)),
    ]).fit(v7_v6_X.loc[v7_rf_fit], v7_v6_y.loc[v7_rf_fit])
    def v7_v6_logit(v7_p):
        v7_p=np.clip(np.asarray(v7_p,dtype=float),1e-6,1-1e-6)
        return np.log(v7_p/(1-v7_p)).reshape(-1,1)
    v7_rf_cal_raw=v7_rf_model.predict_proba(v7_v6_X.loc[v7_rf_cal])[:,1]
    v7_rf_calibrator=LogisticRegression(max_iter=2000,random_state=42).fit(v7_v6_logit(v7_rf_cal_raw),v7_v6_y.loc[v7_rf_cal])
    v7_rf_thr_raw=v7_rf_model.predict_proba(v7_v6_X.loc[v7_rf_thr])[:,1]
    v7_rf_thr_prob=v7_rf_calibrator.predict_proba(v7_v6_logit(v7_rf_thr_raw))[:,1]
    v7_rf_cut_rows=[]
    for v7_cut in np.unique(np.r_[0.0,np.quantile(v7_rf_thr_prob,np.linspace(0,1,301)),0.5,1.0]):
        v7_pred=(v7_rf_thr_prob>=v7_cut).astype("int8")
        v7_tn,v7_fp,v7_fn,v7_tp=confusion_matrix(v7_v6_y.loc[v7_rf_thr],v7_pred,labels=[0,1]).ravel()
        v7_rec=v7_tp/(v7_tp+v7_fn) if v7_tp+v7_fn else 0.0
        v7_spec=v7_tn/(v7_tn+v7_fp) if v7_tn+v7_fp else 0.0
        v7_prec=v7_tp/(v7_tp+v7_fp) if v7_tp+v7_fp else 0.0
        v7_tss=v7_rec+v7_spec-1.0
        v7_rf_cut_rows.append({"threshold":float(v7_cut),"Recall":v7_rec,"Precision":v7_prec,"TSS":v7_tss,"FPR":1-v7_spec,"False_alarms":int(v7_fp),"utility":0.40*v7_rec+0.40*v7_tss+0.20*v7_prec})
    v7_rf_threshold_table=pd.DataFrame(v7_rf_cut_rows).sort_values(["utility","TSS","Recall","Precision"],ascending=False).reset_index(drop=True)
    v7_rf_threshold=float(v7_rf_threshold_table.iloc[0]["threshold"])
    v7_v6_bundle={"model":v7_rf_model,"preprocessing_pipeline":v7_rf_model,"probability_calibrator":v7_rf_calibrator,"feature_columns":list(feature_cols),"operating_threshold":v7_rf_threshold,"hyperparameters":v7_rf_best,"tuning_validation":v7_rf_tuning,"threshold_validation_metrics":v7_rf_threshold_table.iloc[0].to_dict(),"training_period":"2010-2015 with 24-hour purges","calibration_period":"2016 with 24-hour purge","threshold_selection_period":"2017 only with 24-hour purge","target":"label_max; M/X-class flare within 24 hours"}
    v7_v6_rebuilt_path=V7_ROOT/"v6_phase2_reference_regenerated.joblib"
    joblib.dump(v7_v6_bundle,v7_v6_rebuilt_path,compress=3)
    v7_v6_bundle=joblib.load(v7_v6_rebuilt_path)
    v7_v6_artifact_used=str(v7_v6_rebuilt_path)
    v7_rf_probe=v7_v6_X.loc[v7_rf_cal].iloc[:8]
    v7_rf_prob=v7_v6_bundle["probability_calibrator"].predict_proba(v7_v6_logit(v7_v6_bundle["model"].predict_proba(v7_rf_probe)[:,1]))[:,1]
    assert len(v7_rf_prob)==len(v7_rf_probe) and np.isfinite(v7_rf_prob).all()
    v7_rf_tuning.to_csv(V7_ROOT/"v6_phase2_recovery_tuning.csv",index=False)
    v7_rf_threshold_table.to_csv(V7_ROOT/"v6_phase2_recovery_thresholds_2017.csv",index=False)
    print("Regenerated and reload-verified Version 6 reference:",v7_v6_artifact_used,"| features:",len(feature_cols),"| threshold:",round(v7_rf_threshold,6))

# V7 works on the preserved SuryaBench full-disk outcome label, not an active-region target.
print("V6 reference ready:", v7_v6_artifact_used)


In [ ]:
# Query numeric SHARP keywords only; no image segments or FITS files are requested.
# HTTPS is attempted first, with HTTP fallback for environments whose TLS path blocks JSOC.
import json
import requests
import time
from astropy.time import Time

v7_sharp_series = "hmi.sharp_720s"
v7_sharp_cache = V7_CACHE / "sharp_keywords"
v7_sharp_cache.mkdir(exist_ok=True)
v7_sharp_frames, v7_sharp_failures = [], []
v7_sharp_metrics = ["USFLUX", "TOTUSJH", "TOTPOT", "R_VALUE", "TOTUSJZ", "SAVNCPP"]
v7_sharp_id_keys = ["NOAA_AR"]
v7_sharp_keys = ["T_REC", "HARPNUM"] + v7_sharp_id_keys + v7_sharp_metrics
v7_sharp_verified = False
V7_SHARP_MAX_SECONDS = 900
v7_sharp_deadline = time.monotonic() + V7_SHARP_MAX_SECONDS
v7_jsoc_endpoints = [
    "https://jsoc.stanford.edu/cgi-bin/ajax/jsoc_info",
    "http://jsoc.stanford.edu/cgi-bin/ajax/jsoc_info",
]

def v7_query_sharp_year(v7_year):
    v7_params = {
        "op": "rs_list",
        "ds": f"{v7_sharp_series}[][${v7_year}.01.01_TAI-${v7_year+1}.01.01_TAI@6h]".replace("$", ""),
        "key": ",".join(v7_sharp_keys),
    }
    v7_errors = []
    for v7_endpoint in v7_jsoc_endpoints:
        if time.monotonic() >= v7_sharp_deadline:
            break
        try:
            v7_response = requests.get(v7_endpoint, params=v7_params, timeout=(6, 45))
            v7_response.raise_for_status()
            v7_payload = v7_response.json()
            if int(v7_payload.get("status", 0)) != 0:
                raise RuntimeError(str(v7_payload.get("error", v7_payload))[:300])
            v7_columns = {v["name"]: v.get("values", []) for v in v7_payload.get("keywords", [])}
            v7_missing = set(v7_sharp_keys) - set(v7_columns)
            if v7_missing:
                raise RuntimeError(f"JSOC response missing keywords: {sorted(v7_missing)}")
            v7_lengths = {len(v) for v in v7_columns.values()}
            if len(v7_lengths) != 1:
                raise RuntimeError(f"Unequal JSOC keyword lengths: {sorted(v7_lengths)}")
            v7_part = pd.DataFrame(v7_columns)
            if v7_part.empty:
                raise RuntimeError("JSOC returned zero SHARP records")
            return v7_part, None
        except Exception as v7_error:
            v7_errors.append(f"{v7_endpoint.split(':')[0]}: {type(v7_error).__name__}: {str(v7_error)[:180]}")
    return None, " | ".join(v7_errors) or "SHARP query budget exhausted"

for v7_year in range(2010, 2025):
    v7_path = v7_sharp_cache / f"sharp_keywords_{v7_year}.parquet"
    try:
        if v7_path.exists():
            v7_part = pd.read_parquet(v7_path)
            if v7_part.empty or not set(v7_sharp_keys).issubset(v7_part.columns):
                raise RuntimeError("Cached SHARP parquet is empty or missing required numeric keywords")
        elif time.monotonic() < v7_sharp_deadline:
            v7_part, v7_error = v7_query_sharp_year(v7_year)
            if v7_error:
                raise RuntimeError(v7_error)
            v7_part.to_parquet(v7_path, index=False)
        else:
            raise TimeoutError("15-minute SHARP query budget exhausted")
        v7_sharp_frames.append(v7_part)
        print(f"{v7_year}: {len(v7_part):,} SHARP rows", flush=True)
    except Exception as v7_error:
        v7_sharp_failures.append((v7_year, type(v7_error).__name__, str(v7_error)[:400]))
        print(f"{v7_year}: SHARP query/cache unavailable: {type(v7_error).__name__}: {str(v7_error)[:220]}", flush=True)

if not v7_sharp_frames:
    v7_sharp = pd.DataFrame(columns=v7_sharp_keys + ["obs_time", "available_at"])
    v7_sharp_snapshot = pd.DataFrame()
    V7_SOURCE_LIMITATIONS.append("No SHARP keyword records obtained; magnetic features remain missing.")
else:
    v7_sharp = pd.concat(v7_sharp_frames, ignore_index=True).drop_duplicates(["T_REC", "HARPNUM"])
    v7_trec = v7_sharp["T_REC"].astype(str).str.replace(r"_TAI$", "", regex=True).str.replace("_", " ", n=1, regex=False)
    v7_tai = pd.to_datetime(v7_trec, format="%Y.%m.%d %H:%M:%S", errors="coerce")
    v7_keep = v7_tai.notna()
    v7_sharp = v7_sharp.loc[v7_keep].copy()
    v7_tai = v7_tai.loc[v7_keep]
    v7_utc = Time(pd.DatetimeIndex(v7_tai), scale="tai").utc.to_datetime()
    v7_sharp["obs_time"] = pd.to_datetime(v7_utc, utc=True)
    v7_sharp["available_at"] = v7_sharp["obs_time"] + pd.Timedelta(hours=V7_SHARP_DELAY_HOURS)
    for v7_key in ["HARPNUM"] + v7_sharp_metrics + v7_sharp_id_keys:
        v7_sharp[v7_key] = pd.to_numeric(v7_sharp[v7_key], errors="coerce")
        v7_sharp.loc[~np.isfinite(v7_sharp[v7_key]) | (v7_sharp[v7_key].abs() > 1e30), v7_key] = np.nan
    v7_metric_counts = {k: int(v7_sharp[k].notna().sum()) for k in v7_sharp_metrics}
    v7_metric_coverage = {k: float(v7_sharp[k].notna().mean()) for k in v7_sharp_metrics}
    v7_years_present = sorted(v7_sharp["obs_time"].dt.year.dropna().astype(int).unique().tolist())
    v7_span_days = int((v7_sharp["obs_time"].max() - v7_sharp["obs_time"].min()).total_seconds() // 86400) if len(v7_sharp) else 0
    v7_good_metric_count = sum(v7_metric_coverage[k] >= 0.50 for k in v7_sharp_metrics)
    v7_sharp_verified = bool(
        len(v7_sharp) >= 1000 and max(v7_metric_counts.values(), default=0) >= 1000
        and v7_good_metric_count >= 3 and len(v7_years_present) >= 3 and v7_span_days >= 365
    )
    v7_sharp_yearly = v7_sharp.assign(obs_year=v7_sharp["obs_time"].dt.year).groupby("obs_year").agg(
        records=("HARPNUM", "size"),
        finite_USFLUX=("USFLUX", "count"), finite_TOTUSJH=("TOTUSJH", "count"),
        finite_TOTPOT=("TOTPOT", "count"), finite_R_VALUE=("R_VALUE", "count"),
    )
    v7_sharp_yearly.to_csv(V7_ROOT / "sharp_annual_coverage.csv")
    print("SHARP measurement verification:", {
        "verified_historical_coverage": v7_sharp_verified, "records": len(v7_sharp),
        "time_span_days": v7_span_days, "years_present": v7_years_present,
        "finite_measurements": v7_metric_counts, "finite_fraction": {k: round(v, 4) for k, v in v7_metric_coverage.items()},
    })
    if v7_sharp_verified:
        v7_agg = {"sharp_region_count": ("HARPNUM", "nunique")}
        for v7_key in v7_sharp_metrics:
            for v7_stat in ["sum", "max", "mean"]:
                v7_agg[f"sharp_{v7_key}_{v7_stat}"] = (v7_key, v7_stat)
        v7_sharp_snapshot = v7_sharp.sort_values("available_at").groupby("available_at").agg(**v7_agg).sort_index()
    else:
        v7_sharp_snapshot = pd.DataFrame()
        V7_SOURCE_LIMITATIONS.append(
            "SHARP returned records but failed the historical-coverage gate (>=1,000 records, >=3 finite metrics, >=3 years, >=365 days); magnetic predictors are excluded."
        )
        print("LIMITATION:", V7_SOURCE_LIMITATIONS[-1])
print("SHARP whole-Sun snapshots:", v7_sharp_snapshot.shape, "| verified:", v7_sharp_verified)
print("SHARP failures:", v7_sharp_failures)


In [ ]:
# Optional NOAA SRS pipeline: cached reports are reused; network work has a global deadline.
# Failed/partial downloads are not joined into training features unless report coverage is sufficient.
from concurrent.futures import ThreadPoolExecutor, as_completed

v7_srs_cache = V7_CACHE / "noaa_srs"
v7_srs_cache.mkdir(exist_ok=True)
v7_srs_dates = pd.date_range("2010-01-01", "2024-12-31", freq="D", tz="UTC")
v7_srs_base = "https://www.ngdc.noaa.gov/stp/space-weather/swpc-products/daily_reports/solar_region_summaries"
V7_SRS_ENABLED = True
V7_SRS_WALL_BUDGET_SEC = 60
V7_SRS_REQUEST_TIMEOUT = (3, 7)
V7_SRS_MAX_WORKERS = 12
V7_SRS_ATTEMPTS = 2
V7_SRS_MIN_COVERAGE = 0.80

v7_srs_files = []
v7_srs_errors = []
v7_srs_cached = {}
for v7_file in v7_srs_cache.glob("*SRS.txt"):
    if v7_file.stat().st_size > 100:
        try:
            v7_day = pd.to_datetime(v7_file.name[:8], format="%Y%m%d", utc=True)
            v7_srs_cached[v7_day] = v7_file
        except Exception:
            continue

v7_srs_missing = [v7_day for v7_day in v7_srs_dates if v7_day not in v7_srs_cached]
v7_srs_deadline = time.monotonic() + V7_SRS_WALL_BUDGET_SEC

def v7_fetch_srs(v7_day):
    v7_name = v7_day.strftime("%Y%m%d") + "SRS.txt"
    v7_file = v7_srs_cache / v7_name
    v7_url = f"{v7_srs_base}/{v7_day:%Y/%m}/{v7_name}"
    for v7_attempt in range(V7_SRS_ATTEMPTS):
        if time.monotonic() >= v7_srs_deadline:
            return v7_day, None, "global budget exhausted"
        try:
            with requests.get(v7_url, timeout=V7_SRS_REQUEST_TIMEOUT, stream=True) as v7_res:
                if v7_res.status_code == 404:
                    return v7_day, None, "404"
                v7_res.raise_for_status()
                v7_content=bytearray()
                for v7_chunk in v7_res.iter_content(8192):
                    if time.monotonic() >= v7_srs_deadline:
                        raise TimeoutError("SRS global download budget exhausted")
                    v7_content.extend(v7_chunk)
                    if len(v7_content)>1024*1024:
                        raise ValueError("SRS report exceeds one MiB")
            if len(v7_content) <= 100 or b"solar region summary" not in v7_content[:5000].lower():
                raise RuntimeError("Response is empty or not an SRS report")
            v7_part = v7_file.with_suffix(".part")
            v7_part.write_bytes(v7_content)
            v7_part.replace(v7_file)
            return v7_day, v7_file, None
        except Exception as v7_exc:
            if v7_attempt + 1 == V7_SRS_ATTEMPTS:
                return v7_day, None, f"{type(v7_exc).__name__}: {str(v7_exc)[:180]}"
            time.sleep(0.25)
    return v7_day, None, "request failed"

if V7_SRS_ENABLED and v7_srs_missing:
    with ThreadPoolExecutor(max_workers=V7_SRS_MAX_WORKERS) as v7_pool:
        v7_futures = [v7_pool.submit(v7_fetch_srs, v7_day) for v7_day in v7_srs_missing]
        for v7_future in as_completed(v7_futures):
            v7_day, v7_file, v7_err = v7_future.result()
            if v7_file:
                v7_srs_cached[v7_day] = v7_file
            elif v7_err and v7_err != "404" and len(v7_srs_errors) < 100:
                v7_srs_errors.append((str(v7_day.date()), v7_err))

v7_srs_files = sorted(v7_srs_cached.items(), key=lambda item: item[0])
v7_srs_coverage = len(v7_srs_files) / max(1, len(v7_srs_dates))
v7_srs_coverage_by_year = pd.DataFrame({
    "year": range(2010, 2025),
    "reports_cached": [sum(day.year == year for day, _ in v7_srs_files) for year in range(2010, 2025)],
})
v7_srs_coverage_by_year["calendar_days"] = [sum(day.year == year for day in v7_srs_dates) for year in range(2010, 2025)]
v7_srs_coverage_by_year["coverage"] = v7_srs_coverage_by_year["reports_cached"] / v7_srs_coverage_by_year["calendar_days"]
v7_srs_coverage_by_year.to_csv(V7_ROOT / "noaa_srs_annual_coverage.csv", index=False)
v7_srs_available = bool(v7_srs_coverage >= V7_SRS_MIN_COVERAGE)
print("NOAA SRS cache:", len(v7_srs_files), "/", len(v7_srs_dates), "coverage:", round(v7_srs_coverage, 4))
print("NOAA SRS download budget:", V7_SRS_WALL_BUDGET_SEC, "seconds | request timeout:", V7_SRS_REQUEST_TIMEOUT, "| retries:", V7_SRS_ATTEMPTS - 1)
print("NOAA SRS request errors:", len(v7_srs_errors), "| examples:", v7_srs_errors[:5])
print("NOAA SRS coverage by year:")
print(v7_srs_coverage_by_year.to_string(index=False))
if not v7_srs_available:
    V7_SOURCE_LIMITATIONS.append(
        f"NOAA SRS disabled because cached/downloaded report coverage is {v7_srs_coverage:.1%}, below {V7_SRS_MIN_COVERAGE:.0%}; GOES and verified SHARP remain eligible."
    )
    print("LIMITATION:", V7_SOURCE_LIMITATIONS[-1])


In [ ]:
# Parse NOAA SRS active-region tables only when the download coverage gate passes.
# Values become features only after the report date plus 24 hours.
v7_srs_rows = []
v7_srs_region_re = re.compile(r"^\s*(\d{4})\s+([NS]\d{2}[EW]\d{2})\s+(\d{1,3})\s+(\d{1,4})\s+([A-Z]{1,3})\s+(\d{1,2})\s+(\d{1,2})\s+(.+?)\s*$", re.I)

def v7_mag_complexity(v7_text):
    v7_s = re.sub(r"\s+", "", str(v7_text).lower())
    if "beta-gamma-delta" in v7_s or "beta-gammadelta" in v7_s: return 6
    if "gamma-delta" in v7_s: return 5
    if "beta-delta" in v7_s: return 4
    if "beta-gamma" in v7_s: return 3
    if "gamma" in v7_s: return 2
    if "beta" in v7_s: return 2
    if "alpha" in v7_s: return 1
    return np.nan

v7_srs_parse_files = v7_srs_files if v7_srs_available else []
for v7_day, v7_file in v7_srs_parse_files:
    try:
        v7_text = v7_file.read_text(errors="replace")
        v7_regions = []
        v7_in_table = False
        for v7_line in v7_text.splitlines():
            if re.search(r"REGIONS?\s+WITH\s+SUNSPOTS", v7_line, flags=re.I):
                v7_in_table = True
                continue
            if v7_in_table:
                if re.match(r"^\s*(?:IA|II|III)\.", v7_line, flags=re.I):
                    break
                v7_match = v7_srs_region_re.match(v7_line)
                if not v7_match:
                    continue
                v7_no, v7_loc, v7_lon, v7_area, v7_zurich, v7_ll, v7_spots, v7_mag = v7_match.groups()
                v7_regions.append({
                    "region_id": str(v7_no), "area": float(v7_area), "spots": float(v7_spots),
                    "zurich": str(v7_zurich).upper(), "mag_complexity": v7_mag_complexity(v7_mag),
                })
        v7_available = v7_day + pd.Timedelta(hours=24)
        if v7_regions:
            v7_area = np.array([r["area"] for r in v7_regions], dtype=float)
            v7_spots = np.array([r["spots"] for r in v7_regions], dtype=float)
            v7_mag = np.array([r["mag_complexity"] for r in v7_regions], dtype=float)
            v7_zurich = [r["zurich"] for r in v7_regions]
            v7_umbra = np.array([{"A":0,"B":0,"C":1,"D":2,"E":3,"F":4,"H":1}.get(z[0], np.nan) for z in v7_zurich], dtype=float)
            v7_srs_rows.append({
                "available_at": v7_available, "sunspot_region_count": len(v7_regions),
                "sunspot_area_sum": float(np.nansum(v7_area)), "sunspot_area_max": float(np.nanmax(v7_area)),
                "sunspot_spot_count_sum": float(np.nansum(v7_spots)), "sunspot_spot_count_max": float(np.nanmax(v7_spots)),
                "sunspot_mag_complexity_max": float(np.nanmax(v7_mag)) if np.isfinite(v7_mag).any() else np.nan,
                "sunspot_mag_complexity_mean": float(np.nanmean(v7_mag)) if np.isfinite(v7_mag).any() else np.nan,
                "sunspot_complex_region_count": int(np.nansum(v7_mag >= 3)),
                "sunspot_umbra_complexity_max": float(np.nanmax(v7_umbra)) if np.isfinite(v7_umbra).any() else np.nan,
                "sunspot_mcintosh_d_count": sum(z.startswith("D") for z in v7_zurich), "sunspot_mcintosh_e_count": sum(z.startswith("E") for z in v7_zurich), "sunspot_mcintosh_f_count": sum(z.startswith("F") for z in v7_zurich),
                "sunspot_region_ids": sorted({r["region_id"] for r in v7_regions}),
                "sunspot_report_date": v7_day,
            })
        elif v7_in_table and re.search(r"(?im)^\s*(?:NONE|NO REGIONS)\b", re.split(r"(?im)^\s*(?:IA|II|III)\.", re.split(r"(?i)REGIONS?\s+WITH\s+SUNSPOTS", v7_text, maxsplit=1)[-1], maxsplit=1)[0]):
            v7_srs_rows.append({
                "available_at": v7_available, "sunspot_region_count": 0, "sunspot_area_sum": 0.0,
                "sunspot_area_max": 0.0, "sunspot_spot_count_sum": 0.0, "sunspot_spot_count_max": 0.0,
                "sunspot_mag_complexity_max": 0.0, "sunspot_mag_complexity_mean": 0.0,
                "sunspot_complex_region_count": 0, "sunspot_umbra_complexity_max": 0.0,
                "sunspot_region_ids": [], "sunspot_report_date": v7_day,
            })
    except Exception as v7_error:
        print("SRS parse warning:", v7_file.name, type(v7_error).__name__, str(v7_error)[:160])

v7_srs = pd.DataFrame(v7_srs_rows)
v7_srs_parse_coverage = len(v7_srs) / max(1, len(v7_srs_parse_files))
if v7_srs_available and v7_srs_parse_coverage < V7_SRS_MIN_COVERAGE:
    v7_srs = pd.DataFrame()
    v7_srs_available = False
    V7_SOURCE_LIMITATIONS.append(
        f"NOAA SRS disabled because parsed report coverage is {v7_srs_parse_coverage:.1%}, below {V7_SRS_MIN_COVERAGE:.0%}."
    )
if not v7_srs.empty:
    v7_srs = v7_srs.sort_values("available_at").drop_duplicates("available_at", keep="last")
    v7_srs.to_csv(V7_ROOT / "noaa_srs_parsed_features.csv", index=False)
else:
    V7_SOURCE_LIMITATIONS.append("NOAA SRS reports unavailable or insufficiently parseable; sunspot features remain missing.")
print("Parsed NOAA SRS reports:", len(v7_srs), "/", len(v7_srs_parse_files), "| parsed coverage:", round(v7_srs_parse_coverage, 4))
if v7_srs.empty:
    print("LIMITATION:", V7_SOURCE_LIMITATIONS[-1])


In [ ]:
# Causal feature construction and as-of joins.
# NOAA event counts use only M/X events whose peak plus the configured availability delay
# is in a completed hour before the prediction timestamp.
v7_hour_index = v7_goes.index.sort_values().unique()
v7_event_counts = pd.DataFrame(index=v7_hour_index)
for v7_letter in ["M", "X"]:
    if v7_flare_source_ok:
        v7_counts = v7_flare.loc[v7_flare["class_letter"] == v7_letter].groupby("available_hour").size()
        v7_event_counts[f"v7_{v7_letter.lower()}_events"] = v7_counts.reindex(v7_hour_index, fill_value=0).astype(float)
    else:
        v7_event_counts[f"v7_{v7_letter.lower()}_events"] = np.nan
v7_event_counts["v7_mx_events"] = v7_event_counts[["v7_m_events", "v7_x_events"]].sum(axis=1, min_count=2)
v7_history = pd.DataFrame(index=v7_hour_index)
for v7_col in ["v7_m_events", "v7_x_events", "v7_mx_events"]:
    v7_past = v7_event_counts[v7_col].shift(1, fill_value=0) if v7_flare_source_ok else v7_event_counts[v7_col].shift(1)
    for v7_hours in [24, 168, 720]:
        v7_history[f"flare_{v7_col.removeprefix('v7_')}_count_{v7_hours}h"] = v7_past.rolling(v7_hours, min_periods=1).sum()
v7_clock = pd.Series(v7_hour_index, index=v7_hour_index)
v7_last_mx = v7_clock.where(v7_event_counts["v7_mx_events"].gt(0)).ffill().shift(1)
v7_history["flare_hours_since_last_mx"] = (v7_clock - v7_last_mx).dt.total_seconds() / 3600.0
v7_history = v7_history.replace([np.inf, -np.inf], np.nan)

# SHARP whole-Sun aggregation. Each SHARP record is delayed before it can enter the as-of join.
v7_sharp_hourly = pd.DataFrame(index=v7_hour_index)
if not v7_sharp_snapshot.empty:
    v7_sharp_asof = pd.merge_asof(
        pd.DataFrame({"timestamp": v7_hour_index}),
        v7_sharp_snapshot.rename_axis("available_at").reset_index(),
        left_on="timestamp", right_on="available_at", direction="backward",
        tolerance=pd.Timedelta(hours=36),
    ).set_index("timestamp")
    v7_sharp_hourly = v7_sharp_asof.drop(columns=["available_at"], errors="ignore")
    v7_sharp_hourly["sharp_data_age_hours"] = (
        v7_sharp_hourly.index.to_series() - (v7_sharp_asof["available_at"] - pd.Timedelta(hours=V7_SHARP_DELAY_HOURS))
    ).dt.total_seconds() / 3600.0
    v7_level_cols = [c for c in v7_sharp_hourly.columns if c.startswith("sharp_") and c not in ["sharp_region_count", "sharp_data_age_hours"]]
    for v7_col in v7_level_cols:
        v7_signed_log = np.sign(v7_sharp_hourly[v7_col]) * np.log1p(np.abs(v7_sharp_hourly[v7_col]))
        v7_sharp_hourly[f"{v7_col}_signed_log"] = v7_signed_log
        for v7_h in [6, 12, 24]:
            v7_delta = v7_signed_log - v7_signed_log.shift(v7_h)
            v7_sharp_hourly[f"{v7_col}_change_{v7_h}h"] = v7_delta
            v7_sharp_hourly[f"{v7_col}_growth_{v7_h}h"] = (v7_delta / (np.abs(v7_signed_log.shift(v7_h)) + 0.05)).clip(-10, 10)
        v7_sharp_hourly[f"{v7_col}_roll24h_mean"] = v7_signed_log.rolling(24, min_periods=6).mean()
        v7_sharp_hourly[f"{v7_col}_roll24h_std"] = v7_signed_log.rolling(24, min_periods=6).std()

# SRS report features join only after their issue/availability timestamp; missing reports remain NaN.
v7_srs_hourly = pd.DataFrame(index=v7_hour_index)
if not v7_srs.empty:
    v7_srs_numeric = v7_srs.drop(columns=["sunspot_report_date", "sunspot_region_ids"], errors="ignore")
    v7_srs_asof = pd.merge_asof(
        pd.DataFrame({"timestamp": v7_hour_index}),
        v7_srs_numeric.sort_values("available_at"),
        left_on="timestamp", right_on="available_at", direction="backward",
        tolerance=pd.Timedelta(hours=48),
    ).set_index("timestamp")
    v7_srs_hourly = v7_srs_asof.drop(columns=["available_at"], errors="ignore")

# Additional causal evolution, using only as-of observations available at the forecast time.
if not v7_sharp_hourly.empty:
    v8_levels=[c for c in v7_sharp_hourly if c.endswith('_signed_log')]
    v8_extra={}
    for c in v8_levels:
        series=v7_sharp_hourly[c]
        for h in (6,12,24):
            roll=series.rolling(h,min_periods=max(2,h//4))
            v8_extra[f'{c}_rolling{h}h_mean']=roll.mean()
            v8_extra[f'{c}_rolling{h}h_max']=roll.max()
            v8_extra[f'{c}_rolling{h}h_std']=roll.std()
    v7_sharp_hourly=pd.concat([v7_sharp_hourly,pd.DataFrame(v8_extra,index=v7_hour_index)],axis=1)
if not v7_srs_hourly.empty:
    for c in ('sunspot_area_sum','sunspot_spot_count_sum','sunspot_region_count'):
        if c in v7_srs_hourly:
            old=v7_srs_hourly[c].shift(24)
            v7_srs_hourly[c+'_change_24h']=v7_srs_hourly[c]-old
            v7_srs_hourly[c+'_growth_24h']=(v7_srs_hourly[c]-old)/(old.abs()+1.0)
# A trailing 27-day X-ray activity proxy is causal and does not encode a future solar-cycle label.
if 'xrsb_log_mean' in v7_goes:
    v7_history['goes_solar_activity_27d_mean']=v7_goes['xrsb_log_mean'].rolling(648,min_periods=324).mean()
    v7_history['goes_solar_activity_27d_std']=v7_goes['xrsb_log_mean'].rolling(648,min_periods=324).std()
for c in ('xrsa_log_max','xrsb_log_max','xrsa_log_mean','xrsb_log_mean'):
    if c in v7_goes:
        for h in (6,12,24):
            v7_history[f'goes_{c}_lag_{h}h']=v7_goes[c].shift(h)
            v7_history[f'goes_{c}_change_{h}h']=v7_goes[c]-v7_goes[c].shift(h)

# Preserve observed NOAA region IDs for active-region leakage purging; IDs are never model features.
v7_active_region_sets = pd.Series([None] * len(v7_hour_index), index=v7_hour_index, dtype=object)
if not v7_srs.empty and "sunspot_region_ids" in v7_srs:
    v7_srs_activity = v7_srs[["available_at", "sunspot_region_ids"]].sort_values("available_at")
    v7_srs_activity_asof = pd.merge_asof(
        pd.DataFrame({"timestamp": v7_hour_index}), v7_srs_activity,
        left_on="timestamp", right_on="available_at", direction="backward",
        tolerance=pd.Timedelta(hours=48),
    )
    v7_active_region_sets = pd.Series(
        [set(map(str, v)) if isinstance(v, (list, tuple, set)) else None for v in v7_srs_activity_asof["sunspot_region_ids"]],
        index=v7_hour_index, dtype=object,
    )
if not v7_sharp.empty and any(k in v7_sharp.columns for k in ["NOAA_AR", "NOAA_ARS"]):
    def v7_extract_sharp_region_ids(v7_row):
        v7_ids = set()
        for v7_key in ["NOAA_AR", "NOAA_ARS"]:
            if v7_key in v7_row.index and pd.notna(v7_row[v7_key]):
                v7_ids.update(re.findall(r"(?<!\d)\d{4,6}(?!\d)", str(v7_row[v7_key])))
        return v7_ids
    v7_sharp["active_region_ids"] = v7_sharp.apply(v7_extract_sharp_region_ids, axis=1)
    v7_sharp_activity = v7_sharp.groupby("available_at")["active_region_ids"].agg(
        lambda v: sorted(set().union(*(x for x in v if isinstance(x, set))))
    ).reset_index(name="NOAA_ARS")
    v7_sharp_activity_asof = pd.merge_asof(
        pd.DataFrame({"timestamp": v7_hour_index}), v7_sharp_activity,
        left_on="timestamp", right_on="available_at", direction="backward", tolerance=pd.Timedelta(hours=36),
    )
    v7_sharp_ids = [set(map(str, v)) if isinstance(v, (list, tuple, set)) else None for v in v7_sharp_activity_asof["NOAA_ARS"]]
    v7_active_region_sets = pd.Series(
        [(a or set()) | (b or set()) if a is not None or b is not None else None
         for a, b in zip(v7_active_region_sets.to_list(), v7_sharp_ids)], index=v7_hour_index, dtype=object,
    )

v7_features = v7_goes.join(v7_history, how="left").join(v7_sharp_hourly, how="left").join(v7_srs_hourly, how="left")
v7_features = v7_features.replace([np.inf, -np.inf], np.nan)
# Physically interpretable interactions on log-scaled magnetic levels.
for v7_mag_col in ["sharp_USFLUX_sum_signed_log", "sharp_TOTUSJH_sum_signed_log", "sharp_R_VALUE_sum_signed_log"]:
    if v7_mag_col in v7_features:
        for v7_goes_col in ["xrsb_log_max", "xrsa_log_max"]:
            if v7_goes_col in v7_features:
                v7_features[f"interaction_{v7_mag_col}_{v7_goes_col}"] = v7_features[v7_mag_col] * v7_features[v7_goes_col]

v7_data = v7_labels[["label_max"]].join(v7_features, how="inner").sort_index()
v7_data = v7_data.loc[(v7_data.index >= V7_START) & (v7_data.index < V7_END_EXCLUSIVE)]
v7_y = v7_data["label_max"].astype("int8")
v7_all_features = [c for c in v7_data.columns if c != "label_max"]
v7_groups = {
    "GOES": [c for c in v7_all_features if c in feature_cols or c.startswith("goes_")],
    "Magnetic": [c for c in v7_all_features if c.startswith("sharp_")],
    "Sunspot": [c for c in v7_all_features if c.startswith("sunspot_")],
    "Flare history": [c for c in v7_all_features if c.startswith("flare_")],
    "Interaction": [c for c in v7_all_features if c.startswith("interaction_")],
}
assert set(feature_cols).issubset(v7_groups["GOES"])
assert not any(any(t in c.lower() for t in ["label", "target", "max_goes_class", "cumulative_index"]) for c in v7_all_features)
assert all(pd.api.types.is_numeric_dtype(v7_data[c]) for c in v7_all_features)
print("Joined pre-2020 labeled rows/features:", v7_data.shape, "| development target prevalence:", round(float(v7_y.mean()), 4))
print("Feature groups:", {k: len(v) for k, v in v7_groups.items()})
print("Coverage by group:", {k: round(float(v7_data[v].notna().mean().mean()), 3) if v else 0.0 for k, v in v7_groups.items()})

In [ ]:
# Chronological splits with a 24-hour label-overlap purge. Evaluation labels stay unopened.
V7_HORIZON = pd.Timedelta(hours=24)
v7_idx = v7_data.index
v7_dates = {y: pd.Timestamp(f"{y}-01-01", tz="UTC") for y in [2013,2014,2015,2016,2017,2018,2020,2025]}
v7_t_2013,v7_t_2014,v7_t_2015,v7_t_2016,v7_t_2017,v7_t_2018,v7_t_2020,v7_t_2025 = [v7_dates[y] for y in [2013,2014,2015,2016,2017,2018,2020,2025]]
v7_masks = {
 "feature_train":v7_idx < v7_t_2013-V7_HORIZON,
 "feature_validation_2013":(v7_idx>=v7_t_2013)&(v7_idx<v7_t_2014-V7_HORIZON),
 "model_fit":v7_idx < v7_t_2016-V7_HORIZON,
 "calibration_2016":(v7_idx>=v7_t_2016)&(v7_idx<v7_t_2017-V7_HORIZON),
 "threshold_2017":(v7_idx>=v7_t_2017)&(v7_idx<v7_t_2018-V7_HORIZON),
 "negative_diagnostic_2018_2019":(v7_idx>=v7_t_2018)&(v7_idx<v7_t_2020),
}
v7_test_feature_mask=(v7_features.index>=v7_t_2020)&(v7_features.index<v7_t_2025)
v7_test_feature_index=v7_features.index[v7_test_feature_mask]
assert len(v7_test_feature_index)>0, "No 2020-2024 test-period features are available."
V7_ACTIVE_REGION_PURGES=[]
V7_AR_FALLBACK_EMBARGO=pd.Timedelta(days=30)
def v7_purge_shared_regions(train_mask,val_mask,boundary,name):
    train_mask=np.asarray(train_mask,dtype=bool).copy(); val_mask=np.asarray(val_mask,dtype=bool).copy()
    active=v7_active_region_sets.reindex(v7_idx).to_list()
    train_before,val_before=int(train_mask.sum()),int(val_mask.sum())
    train_i,val_i=np.flatnonzero(train_mask),np.flatnonzero(val_mask)
    train_ids=set().union(*(active[i] for i in train_i if isinstance(active[i],set)))
    val_ids=set().union(*(active[i] for i in val_i if isinstance(active[i],set)))
    shared=train_ids&val_ids
    if shared:
        for i in train_i:
            if isinstance(active[i],set) and active[i]&shared: train_mask[i]=False
        for i in val_i:
            if isinstance(active[i],set) and active[i]&shared: val_mask[i]=False
    unknown=np.array([not isinstance(v,set) for v in active],dtype=bool)
    left=(v7_idx>=boundary-V7_AR_FALLBACK_EMBARGO)&(v7_idx<boundary)
    right=(v7_idx>=boundary)&(v7_idx<boundary+V7_AR_FALLBACK_EMBARGO)
    train_mask &= ~(unknown&left); val_mask &= ~(unknown&right)
    V7_ACTIVE_REGION_PURGES.append({"boundary":str(boundary),"split":name,"shared_region_ids":len(shared),"train_rows_removed":train_before-int(train_mask.sum()),"val_rows_removed":val_before-int(val_mask.sum())})
    return train_mask,val_mask,shared
for name,tr,va,boundary in [("feature_selection_2013","feature_train","feature_validation_2013",v7_t_2013),("fit_to_calibration_2016","model_fit","calibration_2016",v7_t_2016),("calibration_to_threshold_2017","calibration_2016","threshold_2017",v7_t_2017)]:
    v7_masks[tr],v7_masks[va],shared=v7_purge_shared_regions(v7_masks[tr],v7_masks[va],boundary,name)
    print(name,"shared NOAA region IDs purged:",len(shared))
v7_cv_masks={}
for year in [2014,2015]:
    start=pd.Timestamp(f"{year}-01-01",tz="UTC"); end=pd.Timestamp(f"{year+1}-01-01",tz="UTC")
    tr=v7_idx<start-V7_HORIZON; va=(v7_idx>=start)&(v7_idx<end-V7_HORIZON)
    tr,va,shared=v7_purge_shared_regions(tr,va,start,f"cv_{year}"); v7_cv_masks[year]=(tr,va)
    print(f"CV {year}: fit={tr.sum():,}, validation={va.sum():,}, shared AR IDs purged={len(shared)}")
for name in ["feature_train","feature_validation_2013","model_fit","calibration_2016","threshold_2017"]:
    part=v7_y.loc[v7_masks[name]]
    print(f"{name:28s} {len(part):,} rows | positives {int(part.sum()):,} | rate {part.mean():.4f}")
    assert len(part)>0 and part.nunique()==2,f"Development split invalid: {name}"
print("Post-freeze data:", {"2018-2019 negative-label diagnostic rows":int(v7_masks["negative_diagnostic_2018_2019"].sum()),"2020-2024 feature rows (outcomes still unread)":len(v7_test_feature_index)})
assert v7_idx[v7_masks["feature_train"]].max()+V7_HORIZON<v7_idx[v7_masks["feature_validation_2013"]].min()
assert v7_idx[v7_masks["model_fit"]].max()+V7_HORIZON<v7_idx[v7_masks["calibration_2016"]].min()
assert v7_idx[v7_masks["calibration_2016"]].max()+V7_HORIZON<v7_idx[v7_masks["threshold_2017"]].min()
for year,(tr,va) in v7_cv_masks.items(): assert v7_idx[tr].max()+V7_HORIZON<v7_idx[va].min(),f"CV {year} horizon overlap"
print("Leakage checks passed. Evaluation labels remain unopened.")


In [ ]:
# Spearman correlation and mutual information use 2010-2012 only.
# Permutation importance fits through 2011 and is scored on held-out 2012; 2013 chooses feature count.
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score,roc_curve
import matplotlib.pyplot as plt
v7_x=v7_data[v7_all_features].replace([np.inf,-np.inf],np.nan)
v7_x_train=v7_x.loc[v7_masks["feature_train"]]; v7_y_train=v7_y.loc[v7_masks["feature_train"]]
v7_x_selval=v7_x.loc[v7_masks["feature_validation_2013"]]; v7_y_selval=v7_y.loc[v7_masks["feature_validation_2013"]]
v7_corr_pool=[c for c in v7_all_features if v7_x_train[c].notna().any() and v7_x_train[c].nunique(dropna=True)>1]
v7_corr=v7_x_train[v7_corr_pool].corr(method="spearman").abs(); v7_upper=v7_corr.where(np.triu(np.ones(v7_corr.shape),k=1).astype(bool))
v7_corr_drop=[c for c in v7_upper.columns if (v7_upper[c]>.985).any()]; v7_corr_keep=[c for c in v7_corr_pool if c not in v7_corr_drop]
print("Spearman screen:",len(v7_corr_pool),"observed;",len(v7_corr_drop),"redundant removed")
v7_2012=pd.Timestamp("2012-01-01",tz="UTC"); v7_2013=pd.Timestamp("2013-01-01",tz="UTC")
v7_rank_fit_full=v7_idx<v7_2012-V7_HORIZON
v7_rank_val_full=(v7_idx>=v7_2012)&(v7_idx<v7_2013-V7_HORIZON)
v7_rank_fit_full,v7_rank_val_full,v7_rank_shared=v7_purge_shared_regions(v7_rank_fit_full,v7_rank_val_full,v7_2012,"feature_ranking_2012")
v7_rfit=v7_x_train.index.isin(v7_idx[v7_rank_fit_full]); v7_rval=v7_x_train.index.isin(v7_idx[v7_rank_val_full])
v7_rank_fit_y=v7_y_train.loc[v7_rfit]; v7_rank_val_y=v7_y_train.loc[v7_rval]
assert v7_rank_fit_y.nunique()==2 and v7_rank_val_y.nunique()==2
v7_rank_imp=SimpleImputer(strategy="median",keep_empty_features=True)
v7_rank_fit=v7_rank_imp.fit_transform(v7_x_train.loc[v7_rfit,v7_corr_keep]); v7_rank_val=v7_rank_imp.transform(v7_x_train.loc[v7_rval,v7_corr_keep])
v7_mi_score=pd.Series(mutual_info_classif(v7_rank_fit,v7_rank_fit_y,random_state=V7_RANDOM_SEED),index=v7_corr_keep,name="mutual_information")
print("Supervised feature ranking fit rows:",len(v7_rank_fit_y),"| purged shared IDs:",len(v7_rank_shared),"| 2012 holdout rows:",len(v7_rank_val_y))
v7_perm_model=RandomForestClassifier(n_estimators=300,max_depth=12,min_samples_leaf=15,max_features="sqrt",class_weight="balanced_subsample",n_jobs=4,random_state=V7_RANDOM_SEED).fit(v7_rank_fit,v7_rank_fit_y)
if len(v7_rank_val_y)>12000:
    take=np.random.default_rng(V7_RANDOM_SEED).choice(len(v7_rank_val_y),12000,replace=False); v7_rank_val=v7_rank_val[take]; v7_rank_val_y=v7_rank_val_y.iloc[take]
v7_perm=permutation_importance(v7_perm_model,v7_rank_val,v7_rank_val_y,scoring="average_precision",n_repeats=5,random_state=V7_RANDOM_SEED,n_jobs=2)
v7_perm_score=pd.Series(np.maximum(v7_perm.importances_mean,0),index=v7_corr_keep,name="permutation_ap_2012")
try:
    import shap
    ix=np.random.default_rng(V7_RANDOM_SEED).choice(len(v7_rank_fit),min(1000,len(v7_rank_fit)),replace=False)
    vals=shap.TreeExplainer(v7_perm_model).shap_values(v7_rank_fit[ix])
    if isinstance(vals,list): arr=np.asarray(vals[-1])
    else:
        arr=np.asarray(vals)
        if arr.ndim==3: arr=arr[:,:,-1]
    v7_shap_score=pd.Series(np.abs(arr).mean(axis=0),index=v7_corr_keep,name="mean_abs_shap")
    shap.summary_plot(arr,v7_rank_fit[ix],feature_names=v7_corr_keep,show=False,max_display=20)
    plt.tight_layout(); plt.savefig(V7_ROOT/"feature_selection_shap_training_only.png",dpi=160,bbox_inches="tight"); plt.show()
except Exception as e:
    v7_shap_score=pd.Series(0.,index=v7_corr_keep,name="mean_abs_shap")
    V7_SOURCE_LIMITATIONS.append(f"SHAP unavailable: {type(e).__name__}: {str(e)[:180]}")
    print("SHAP unavailable; other feature methods remain active:",V7_SOURCE_LIMITATIONS[-1])
def v7_unit_rank(s):
    s=s.reindex(v7_corr_keep).fillna(0); span=s.max()-s.min()
    return (s-s.min())/span if span>0 else s*0
v7_rank=pd.DataFrame({"mutual_information":v7_mi_score,"permutation_ap_2012":v7_perm_score,"mean_abs_shap":v7_shap_score})
v7_rank['model_importance']=v7_perm_model.feature_importances_
try:
    from sklearn.feature_selection import RFE
    rfe_pool=v7_rank.sort_values('model_importance',ascending=False).head(min(40,len(v7_rank))).index.tolist()
    rfe_ix=[v7_corr_keep.index(c) for c in rfe_pool]
    rfe=RFE(RandomForestClassifier(n_estimators=50,max_depth=8,min_samples_leaf=15,n_jobs=4,random_state=V7_RANDOM_SEED),n_features_to_select=max(1,len(rfe_pool)//2),step=.5)
    rfe.fit(v7_rank_fit[:,rfe_ix],v7_rank_fit_y)
    v7_rank['rfe_selected']=0.0; v7_rank.loc[rfe_pool,'rfe_selected']=rfe.support_.astype(float)
except Exception as e:
    v7_rank['rfe_selected']=0.0
    V7_SOURCE_LIMITATIONS.append(f'RFE unavailable: {type(e).__name__}: {e}')

v7_rank["combined_rank_score"]=.25*v7_unit_rank(v7_rank["mutual_information"])+.30*v7_unit_rank(v7_rank["permutation_ap_2012"])+.20*v7_unit_rank(v7_rank["mean_abs_shap"])+.20*v7_unit_rank(v7_rank["model_importance"])+.05*v7_rank["rfe_selected"]
v7_rank=v7_rank.sort_values("combined_rank_score",ascending=False); v7_rank.to_csv(V7_ROOT/"feature_importance_selection.csv")
v7_rank.head(25).sort_values("combined_rank_score").plot.barh(y="combined_rank_score",figsize=(9,8),legend=False,title="Training-only V7 feature ranking")
plt.tight_layout(); plt.savefig(V7_ROOT/"feature_ranking.png",dpi=160,bbox_inches="tight"); plt.show()
v7_ranked_features=v7_rank.index.tolist(); sizes=sorted(set(min(len(v7_ranked_features),n) for n in [25,50,75,len(v7_ranked_features)] if n>0))
v7_feature_sets={f"top_{n}":v7_ranked_features[:n] for n in sizes}; v7_subset_rows=[]
# Every subset comparison ranks features using only the training portion of that fold.
for fold_year in (2012,2013):
    boundary=pd.Timestamp(f'{fold_year}-01-01',tz='UTC'); end=pd.Timestamp(f'{fold_year+1}-01-01',tz='UTC')
    tr=(v7_idx<boundary-V7_HORIZON); va=(v7_idx>=boundary)&(v7_idx<end-V7_HORIZON)
    tr,va,_=v7_purge_shared_regions(tr,va,boundary,f'subset_cv_{fold_year}')
    xtr0=v7_x.loc[tr]; xva0=v7_x.loc[va]; yt=v7_y.loc[tr]; yv=v7_y.loc[va]
    assert yt.nunique()==2 and yv.nunique()==2
    pool=[c for c in v7_all_features if xtr0[c].notna().any() and xtr0[c].nunique(dropna=True)>1]
    corr=xtr0[pool].corr(method='spearman').abs(); upper=corr.where(np.triu(np.ones(corr.shape),k=1).astype(bool))
    pool=[c for c in pool if not (upper[c]>.985).any()]
    imp0=SimpleImputer(strategy='median',keep_empty_features=True); a=imp0.fit_transform(xtr0[pool]); b=imp0.transform(xva0[pool])
    mi=mutual_info_classif(a,yt,random_state=V7_RANDOM_SEED)
    fm=RandomForestClassifier(n_estimators=120,max_depth=10,min_samples_leaf=15,class_weight='balanced_subsample',n_jobs=4,random_state=V7_RANDOM_SEED).fit(a,yt)
    order=np.argsort(-(pd.Series(mi).rank(pct=True).to_numpy()+pd.Series(fm.feature_importances_).rank(pct=True).to_numpy()))
    for count in sizes:
        chosen=order[:min(count,len(order))]
        m=RandomForestClassifier(n_estimators=160,max_depth=12,min_samples_leaf=15,class_weight='balanced_subsample',n_jobs=4,random_state=V7_RANDOM_SEED).fit(a[:,chosen],yt)
        prob=m.predict_proba(b[:,chosen])[:,1]; fpr,tpr,_=roc_curve(yv,prob)
        ap=average_precision_score(yv,prob); tss=float(np.max(tpr-fpr))
        v7_subset_rows.append({'subset':f'top_{count}','features':count,'fold':fold_year,'PR_AUC':ap,'max_TSS':tss,'selection_score':.5*ap+.5*tss})
v7_subset_folds=pd.DataFrame(v7_subset_rows)
v7_subset_folds.to_csv(V7_ROOT/'chronological_feature_subset_folds.csv',index=False)
v7_subset_table=v7_subset_folds.groupby(['subset','features'],as_index=False).agg(selection_score=('selection_score','mean'),score_std=('selection_score','std'))
v7_subset_table=v7_subset_table.sort_values('features').reset_index(drop=True)
best=float(v7_subset_table['selection_score'].max())
# Prefer the smallest set within 0.01 absolute utility of the best mean score.
v7_selected_set_name=str(v7_subset_table.loc[v7_subset_table.selection_score>=best-0.01].iloc[0]['subset'])
v7_selected_features=v7_feature_sets[v7_selected_set_name]
v7_subset_table.to_csv(V7_ROOT/'chronological_feature_subset_validation_2013.csv',index=False); display(v7_subset_table)
print('Subset selected from 2012/2013 chronological folds:',v7_selected_set_name,len(v7_selected_features))


In [ ]:
# Optuna chronological hyperparameter search for four model families.
import optuna
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_curve
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)
V7_OPTUNA_TRIALS = 12
v7_cv_years = [2014, 2015]

def v7_estimator(v7_family, v7_params, v7_target):
    v7_pos_weight = float((v7_target == 0).sum() / max(1, (v7_target == 1).sum()))
    if v7_family == "Random Forest":
        v7_est = RandomForestClassifier(
            n_estimators=v7_params["n_estimators"], max_depth=v7_params["max_depth"],
            min_samples_leaf=v7_params["min_samples_leaf"], max_features=v7_params["max_features"],
            class_weight="balanced_subsample", n_jobs=4, random_state=V7_RANDOM_SEED,
        )
    elif v7_family == "XGBoost":
        v7_est = XGBClassifier(
            n_estimators=v7_params["n_estimators"], max_depth=v7_params["max_depth"],
            learning_rate=v7_params["learning_rate"], min_child_weight=v7_params["min_child_weight"],
            subsample=v7_params["subsample"], colsample_bytree=v7_params["colsample_bytree"],
            reg_lambda=v7_params["reg_lambda"], scale_pos_weight=v7_pos_weight,
            objective="binary:logistic", eval_metric="logloss", tree_method="hist",
            n_jobs=4, random_state=V7_RANDOM_SEED,
        )
    elif v7_family == "LightGBM":
        v7_est = LGBMClassifier(
            n_estimators=v7_params["n_estimators"], num_leaves=v7_params["num_leaves"],
            learning_rate=v7_params["learning_rate"], min_child_samples=v7_params["min_child_samples"],
            subsample=v7_params["subsample"], subsample_freq=1, colsample_bytree=v7_params["colsample_bytree"],
            reg_lambda=v7_params["reg_lambda"], scale_pos_weight=v7_pos_weight,
            verbosity=-1, n_jobs=4, random_state=V7_RANDOM_SEED,
        )
    elif v7_family == "CatBoost":
        v7_est = CatBoostClassifier(
            iterations=v7_params["iterations"], depth=v7_params["depth"],
            learning_rate=v7_params["learning_rate"], l2_leaf_reg=v7_params["l2_leaf_reg"],
            auto_class_weights="Balanced", verbose=False, allow_writing_files=False,
            thread_count=4, random_seed=V7_RANDOM_SEED,
        )
    else:
        raise ValueError(v7_family)
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
        ("model", v7_est),
    ])

def v7_suggest_params(v7_trial, v7_family):
    if v7_family == "Random Forest":
        return {
            "n_estimators": v7_trial.suggest_int("n_estimators", 200, 500, step=100),
            "max_depth": v7_trial.suggest_categorical("max_depth", [8, 12, 16, None]),
            "min_samples_leaf": v7_trial.suggest_int("min_samples_leaf", 5, 40, log=True),
            "max_features": v7_trial.suggest_categorical("max_features", ["sqrt", 0.5, 0.8]),
        }
    if v7_family == "XGBoost":
        return {
            "n_estimators": v7_trial.suggest_int("n_estimators", 200, 600, step=100),
            "max_depth": v7_trial.suggest_int("max_depth", 2, 6),
            "learning_rate": v7_trial.suggest_float("learning_rate", 0.015, 0.12, log=True),
            "min_child_weight": v7_trial.suggest_float("min_child_weight", 2, 30, log=True),
            "subsample": v7_trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree": v7_trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_lambda": v7_trial.suggest_float("reg_lambda", 1.0, 15.0, log=True),
        }
    if v7_family == "LightGBM":
        return {
            "n_estimators": v7_trial.suggest_int("n_estimators", 200, 600, step=100),
            "num_leaves": v7_trial.suggest_int("num_leaves", 7, 31),
            "learning_rate": v7_trial.suggest_float("learning_rate", 0.015, 0.12, log=True),
            "min_child_samples": v7_trial.suggest_int("min_child_samples", 40, 200, step=20),
            "subsample": v7_trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree": v7_trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_lambda": v7_trial.suggest_float("reg_lambda", 1.0, 15.0, log=True),
        }
    return {
        "iterations": v7_trial.suggest_int("iterations", 250, 650, step=100),
        "depth": v7_trial.suggest_int("depth", 4, 8),
        "learning_rate": v7_trial.suggest_float("learning_rate", 0.015, 0.12, log=True),
        "l2_leaf_reg": v7_trial.suggest_float("l2_leaf_reg", 1.0, 15.0, log=True),
    }

v7_optuna_studies, v7_best_params, v7_cv_rows = {}, {}, []
for v7_family in ["Random Forest", "XGBoost", "LightGBM", "CatBoost"]:
    def v7_objective(v7_trial, v7_family=v7_family):
        v7_params = v7_suggest_params(v7_trial, v7_family)
        v7_scores = []
        for v7_year in v7_cv_years:
            v7_start = pd.Timestamp(f"{v7_year}-01-01", tz="UTC")
            v7_next = pd.Timestamp(f"{v7_year+1}-01-01", tz="UTC")
            v7_fit_mask, v7_val_mask = v7_cv_masks[v7_year]
            v7_yfit, v7_yval = v7_y.loc[v7_fit_mask], v7_y.loc[v7_val_mask]
            if v7_yfit.nunique() < 2 or v7_yval.nunique() < 2:
                continue
            v7_model = v7_estimator(v7_family, v7_params, v7_yfit)
            v7_model.fit(v7_x.loc[v7_fit_mask, v7_selected_features], v7_yfit)
            v7_prob = v7_model.predict_proba(v7_x.loc[v7_val_mask, v7_selected_features])[:, 1]
            v7_scores.append(average_precision_score(v7_yval, v7_prob))
        if not v7_scores:
            raise optuna.TrialPruned()
        return float(np.mean(v7_scores))
    v7_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=V7_RANDOM_SEED))
    v7_study.optimize(v7_objective, n_trials=V7_OPTUNA_TRIALS, show_progress_bar=True, n_jobs=1)
    v7_optuna_studies[v7_family] = v7_study
    v7_best_params[v7_family] = dict(v7_study.best_params)
    v7_cv_rows.append({
        "Model": v7_family, "Mean_CV_PR_AUC_2014_2015": float(v7_study.best_value),
        "Best_parameters": json.dumps(v7_study.best_params, sort_keys=True),
        "trials": len(v7_study.trials),
    })
    print(v7_family, "CV mean PR-AUC:", round(v7_study.best_value, 4), "|", v7_study.best_params, flush=True)

v7_cv_table = pd.DataFrame(v7_cv_rows).sort_values("Mean_CV_PR_AUC_2014_2015", ascending=False).reset_index(drop=True)
v7_cv_table.to_csv(V7_ROOT / "optuna_chronological_cv.csv", index=False)
v7_champion_family = str(v7_cv_table.iloc[0]["Model"])
print("Pre-test champion family selected on 2014–2015 chronological CV:", v7_champion_family)
print("Optuna trials per family:", V7_OPTUNA_TRIALS, "| trials did not access 2016–2024; evaluation labels remained unopened.");

In [ ]:
# Final refit, probability calibration, and operational thresholds.
# Model choice comes from chronological 2014–2015 Optuna CV; calibration and thresholds use separate 2016/2017 periods.
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score, brier_score_loss
from sklearn.linear_model import LogisticRegression
import joblib

v7_fit_X = v7_x.loc[v7_masks["model_fit"], v7_selected_features]
v7_fit_y = v7_y.loc[v7_masks["model_fit"]]
v7_cal_X = v7_x.loc[v7_masks["calibration_2016"], v7_selected_features]
v7_cal_y = v7_y.loc[v7_masks["calibration_2016"]]
v7_thr_X = v7_x.loc[v7_masks["threshold_2017"], v7_selected_features]
v7_thr_y = v7_y.loc[v7_masks["threshold_2017"]]
v7_test_X = v7_features.loc[v7_test_feature_index, v7_selected_features]
v7_negative_X = v7_x.loc[v7_masks["negative_diagnostic_2018_2019"], v7_selected_features]
assert v7_fit_X.index.max() + V7_HORIZON < v7_cal_X.index.min()
assert v7_cal_X.index.max() + V7_HORIZON < v7_thr_X.index.min()
assert v7_thr_X.index.max() + V7_HORIZON < v7_negative_X.index.min()
assert v7_negative_X.index.max() < v7_test_X.index.min()
assert v7_test_X.index.max() < pd.Timestamp("2025-01-01", tz="UTC")

def v7_logit(v7_p):
    v7_p = np.clip(np.asarray(v7_p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(v7_p / (1 - v7_p)).reshape(-1, 1)

def v7_fit_calibrator(v7_model, v7_xcal, v7_ycal):
    v7_raw = v7_model.predict_proba(v7_xcal)[:, 1]
    v7_cal = LogisticRegression(C=1.0, max_iter=2000, random_state=V7_RANDOM_SEED)
    v7_cal.fit(v7_logit(v7_raw), v7_ycal)
    return v7_cal

def v7_calibrated_probability(v7_model, v7_cal, v7_xnew):
    v7_raw = v7_model.predict_proba(v7_xnew)[:, 1]
    return v7_cal.predict_proba(v7_logit(v7_raw))[:, 1]

def v7_threshold_table(v7_ytrue, v7_prob):
    v7_ytrue = np.asarray(v7_ytrue, dtype="int8")
    v7_prob = np.asarray(v7_prob, dtype=float)
    v7_grid = np.unique(np.r_[0.0, np.quantile(v7_prob, np.linspace(0, 1, 501)), 0.5, 1.0])
    v7_rows = []
    for v7_cut in v7_grid:
        v7_pred = (v7_prob >= v7_cut).astype("int8")
        v7_tn, v7_fp, v7_fn, v7_tp = confusion_matrix(v7_ytrue, v7_pred, labels=[0, 1]).ravel()
        v7_recall = v7_tp / (v7_tp + v7_fn) if v7_tp + v7_fn else 0.0
        v7_spec = v7_tn / (v7_tn + v7_fp) if v7_tn + v7_fp else 0.0
        v7_precision = v7_tp / (v7_tp + v7_fp) if v7_tp + v7_fp else 0.0
        v7_rows.append({
            "threshold": float(v7_cut), "Recall": v7_recall, "Precision": v7_precision,
            "TSS": v7_recall + v7_spec - 1, "FPR": 1 - v7_spec,
            "False_alarms": int(v7_fp), "TP": int(v7_tp),
        })
    return pd.DataFrame(v7_rows)

def v7_choose_thresholds(v7_ytrue, v7_prob):
    v7_tab = v7_threshold_table(v7_ytrue, v7_prob)
    v7_tab["utility"] = 0.40 * v7_tab["Recall"] + 0.40 * v7_tab["TSS"] + 0.20 * v7_tab["Precision"]
    v7_operating = v7_tab.sort_values(["utility", "TSS", "Recall", "Precision"], ascending=False).iloc[0]
    v7_recall_ok = v7_tab.loc[v7_tab["Recall"] >= 0.90].sort_values(["Precision", "TSS", "threshold"], ascending=False)
    v7_precision_ok = v7_tab.loc[v7_tab["Precision"] >= 0.70].sort_values(["Recall", "TSS", "Precision"], ascending=False)
    return {
        "operating": float(v7_operating["threshold"]),
        "watch_recall_90": float(v7_recall_ok.iloc[0]["threshold"]),
        "high_precision_70": float(v7_precision_ok.iloc[0]["threshold"]) if len(v7_precision_ok) else None,
        "validation_table": v7_tab,
    }

v7_final_models, v7_calibrators, v7_thresholds, v7_thr_probabilities, v7_test_probabilities, v7_negative_probabilities = {}, {}, {}, {}, {}, {}
for v7_family in ["Random Forest", "XGBoost", "LightGBM", "CatBoost"]:
    print("Final chronological fit:", v7_family, flush=True)
    v7_model = v7_estimator(v7_family, v7_best_params[v7_family], v7_fit_y)
    v7_model.fit(v7_fit_X, v7_fit_y)
    v7_cal = v7_fit_calibrator(v7_model, v7_cal_X, v7_cal_y)
    v7_pthr = v7_calibrated_probability(v7_model, v7_cal, v7_thr_X)
    v7_cuts = v7_choose_thresholds(v7_thr_y, v7_pthr)
    v7_final_models[v7_family] = v7_model
    v7_calibrators[v7_family] = v7_cal
    v7_thresholds[v7_family] = {k: v for k, v in v7_cuts.items() if k != "validation_table"}
    v7_cuts["validation_table"].to_csv(V7_ROOT / f"thresholds_2017_{v7_family.lower().replace(' ', '_')}.csv", index=False)
    v7_thr_probabilities[v7_family] = v7_pthr
    v7_test_probabilities[v7_family] = v7_calibrated_probability(v7_model, v7_cal, v7_test_X)
    v7_negative_probabilities[v7_family] = v7_calibrated_probability(v7_model, v7_cal, v7_negative_X)
    print(v7_family, "2017 thresholds:", v7_thresholds[v7_family], flush=True)

# Equal-weight soft voting across calibrated probabilities. Threshold is still selected only on 2017.
v7_thr_probabilities["Soft-voting ensemble"] = np.mean(np.column_stack(list(v7_thr_probabilities.values())), axis=1)
v7_test_probabilities["Soft-voting ensemble"] = np.mean(np.column_stack(list(v7_test_probabilities.values())), axis=1)
v7_negative_probabilities["Soft-voting ensemble"] = np.mean(np.column_stack(list(v7_negative_probabilities.values())), axis=1)
v7_ensemble_cuts = v7_choose_thresholds(v7_thr_y, v7_thr_probabilities["Soft-voting ensemble"])
v7_thresholds["Soft-voting ensemble"] = {k: v for k, v in v7_ensemble_cuts.items() if k != "validation_table"}
v7_ensemble_cuts["validation_table"].to_csv(V7_ROOT / "thresholds_2017_soft_voting_ensemble.csv", index=False)
print("Soft-voting ensemble 2017 thresholds:", v7_thresholds["Soft-voting ensemble"])

v7_champion_model = v7_final_models[v7_champion_family]
v7_champion_calibrator = v7_calibrators[v7_champion_family]
v7_champion_thresholds = v7_thresholds[v7_champion_family]
print("Frozen pre-test champion:", v7_champion_family, "| operating threshold:", v7_champion_thresholds["operating"])

In [ ]:
# Ablations share a frozen model family/parameter set; each gets its own 2016 calibrator and 2017 threshold.
def v8_ablation_screen(cols):
    cols=[c for c in cols if c in v7_corr_pool]
    if not cols: return []
    corr=v7_x_train[cols].corr(method="spearman").abs()
    upper=corr.where(np.triu(np.ones(corr.shape),k=1).astype(bool))
    return [c for c in cols if not (upper[c]>.985).any()]
v7_available_groups={k:v8_ablation_screen(v) for k,v in v7_groups.items()}
v7_ablation_sets={}
for group in ["GOES","Magnetic","Sunspot"]:
    if v7_available_groups.get(group): v7_ablation_sets[f"{group}-only"]=v7_available_groups[group]
if v7_available_groups.get("GOES") and v7_available_groups.get("Magnetic"):
    v7_ablation_sets["GOES + magnetic"]=v7_available_groups["GOES"]+v7_available_groups["Magnetic"]
if v7_available_groups.get("GOES") and v7_available_groups.get("Sunspot"):
    v7_ablation_sets["GOES + sunspot"]=v7_available_groups["GOES"]+v7_available_groups["Sunspot"]
v7_ablation_sets["All combined"]=list(v7_corr_keep)
v7_ablation_models,v7_ablation_calibrators,v7_ablation_thresholds={},{},{}
v7_ablation_thr_probabilities,v7_ablation_test_probabilities,v7_ablation_negative_probabilities={},{},{}
for name,cols in v7_ablation_sets.items():
    cols=list(dict.fromkeys(cols))
    if not cols: continue
    print("Ablation fit:",name,"features:",len(cols),flush=True)
    model=v7_estimator(v7_champion_family,v7_best_params[v7_champion_family],v7_fit_y).fit(v7_x.loc[v7_masks["model_fit"],cols],v7_fit_y)
    cal=v7_fit_calibrator(model,v7_x.loc[v7_masks["calibration_2016"],cols],v7_cal_y)
    ptr=v7_calibrated_probability(model,cal,v7_x.loc[v7_masks["threshold_2017"],cols])
    cut=v7_choose_thresholds(v7_thr_y,ptr)
    v7_ablation_models[name]=model; v7_ablation_calibrators[name]=cal
    v7_ablation_thresholds[name]={k:v for k,v in cut.items() if k!="validation_table"}
    v7_ablation_thr_probabilities[name]=ptr
    cut["validation_table"].to_csv(V7_ROOT/f"thresholds_2017_ablation_{name.lower().replace(' ','_').replace('+','plus')}.csv",index=False)
    v7_ablation_test_probabilities[name]=v7_calibrated_probability(model,cal,v7_features.loc[v7_test_X.index,cols])
    v7_ablation_negative_probabilities[name]=v7_calibrated_probability(model,cal,v7_x.loc[v7_masks["negative_diagnostic_2018_2019"],cols])
print("Ablation feature counts:",{k:len(v) for k,v in v7_ablation_sets.items()})


In [ ]:
# Evaluation labels are opened only after every model, feature, calibrator, threshold, and baseline reference is frozen.
import joblib
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,roc_auc_score,average_precision_score,brier_score_loss,log_loss
v7_v6_phase2_path=Path("/kaggle/working/solaris_ai_random_forest_phase2.joblib")
v7_v6_opt_path=Path("/kaggle/working/solaris_ai_random_forest_optimized.joblib")
if "v7_v6_bundle" in globals() and v7_v6_bundle is not None:
    pass
elif v7_v6_phase2_path.exists():
    v7_v6_bundle=joblib.load(v7_v6_phase2_path); v7_v6_artifact_used=str(v7_v6_phase2_path)
elif v7_v6_opt_path.exists():
    v7_v6_bundle=joblib.load(v7_v6_opt_path); v7_v6_artifact_used=str(v7_v6_opt_path)
else:
    raise RuntimeError("Run the Version 8 V6-artifact recovery cell before opening evaluation outcomes.")

# Only timestamp values were read earlier. Load label_max for 2020-2024 now, after all choices are frozen.
v7_eval_row_ids=np.flatnonzero((v7_label_row_times>=v7_t_2020)&(v7_label_row_times<v7_t_2025))
v7_eval_row_id_set=set(map(int,v7_eval_row_ids))
v7_test_label_frame=pd.read_csv(
    v7_label_csv,usecols=["timestamp","label_max"],
    skiprows=lambda row: row>0 and (row-1) not in v7_eval_row_id_set,
    parse_dates=["timestamp"],
)
assert len(v7_test_label_frame)==len(v7_eval_row_ids), "Test label row selection did not match the timestamp index."
v7_test_label_frame["timestamp"]=pd.to_datetime(v7_test_label_frame["timestamp"],utc=True)
v7_test_y=v7_test_label_frame.drop_duplicates("timestamp").set_index("timestamp")["label_max"].sort_index().astype("int8")
v7_test_eval_index=v7_test_y.index.intersection(v7_test_X.index).sort_values()
v7_label_coverage=len(v7_test_eval_index)/max(1,len(v7_test_y))
assert v7_label_coverage>=0.95, f"Only {v7_label_coverage:.1%} of test labels align with hourly feature rows."
v7_test_positions=v7_test_X.index.get_indexer(v7_test_eval_index)
assert (v7_test_positions>=0).all()
for v7_name in list(v7_test_probabilities):
    v7_values=np.asarray(v7_test_probabilities[v7_name])
    assert len(v7_values)==len(v7_test_X), f"Prediction alignment error for {v7_name}."
    v7_test_probabilities[v7_name]=v7_values[v7_test_positions]
for v7_name in list(v7_ablation_test_probabilities):
    v7_values=np.asarray(v7_ablation_test_probabilities[v7_name])
    assert len(v7_values)==len(v7_test_X), f"Ablation prediction alignment error for {v7_name}."
    v7_ablation_test_probabilities[v7_name]=v7_values[v7_test_positions]
v7_test_X=v7_test_X.loc[v7_test_eval_index]
v7_test_y=v7_test_y.loc[v7_test_eval_index]
v7_negative_y=v7_y.loc[v7_masks["negative_diagnostic_2018_2019"]]
assert v7_test_y.nunique()==2,"2020-2024 descriptive comparison must contain both classes"
print("Opened post-freeze 2020-2024 labels:",len(v7_test_y),"| feature/label coverage:",round(v7_label_coverage,4))
def v7_metrics(y,p,cut):
    y=np.asarray(y,dtype="int8"); p=np.clip(np.asarray(p,dtype=float),1e-7,1-1e-7); pred=(p>=float(cut)).astype("int8")
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel(); rec=recall_score(y,pred,zero_division=0); spec=tn/(tn+fp) if tn+fp else 0.
    bins=np.minimum((p*10).astype(int),9); ece=sum(float((bins==b).mean())*abs(float(p[bins==b].mean())-float(y[bins==b].mean())) for b in range(10) if np.any(bins==b))
    both=len(np.unique(y))==2
    return {"N":len(y),"Prevalence":float(y.mean()),"Threshold":float(cut),"Accuracy":accuracy_score(y,pred),"Precision":precision_score(y,pred,zero_division=0),"Recall":rec,"F1":f1_score(y,pred,zero_division=0),"TSS":rec+spec-1,"FPR":1-spec,"False_alarms_FP":int(fp),"ROC_AUC":roc_auc_score(y,p) if both else np.nan,"PR_AUC":average_precision_score(y,p) if y.sum() else np.nan,"Brier":brier_score_loss(y,p),"Log_loss":log_loss(y,p,labels=[0,1]),"ECE_10_bins":ece,"TN":int(tn),"FP":int(fp),"FN":int(fn),"TP":int(tp)}
v7_final_results=[]; v7_probability_map={}
for family in ["Random Forest","XGBoost","LightGBM","CatBoost"]:
    prob=v7_test_probabilities[family]; v7_probability_map[family]=prob; cut=v7_thresholds[family]["operating"]
    v7_final_results += [{"Model":family,"Decision":"2017 utility threshold",**v7_metrics(v7_test_y,prob,cut)},{"Model":family,"Decision":"fixed 0.50",**v7_metrics(v7_test_y,prob,.5)}]
v7_probability_map["Soft-voting ensemble"]=v7_test_probabilities["Soft-voting ensemble"]
for decision,cut in [("2017 utility threshold",v7_thresholds["Soft-voting ensemble"]["operating"]),("fixed 0.50",.5)]:
    v7_final_results.append({"Model":"Soft-voting ensemble","Decision":decision,**v7_metrics(v7_test_y,v7_test_probabilities["Soft-voting ensemble"],cut)})
# The Version 6 Phase 2 bundle was verified or regenerated before test-label loading.
assert v7_v6_bundle is not None and v7_v6_artifact_used
v7_v6_cols=list(v7_v6_bundle["feature_columns"]); assert set(v7_v6_cols).issubset(v7_x.columns)
v7_v6_Xtest=goesh.loc[v7_test_X.index,v7_v6_cols]
v7_v6_test_raw=v7_v6_bundle["model"].predict_proba(v7_v6_Xtest)[:,1]
v7_v6_prob=v7_v6_bundle["probability_calibrator"].predict_proba(v7_logit(v7_v6_test_raw))[:,1]
v7_v6_cut=float(v7_v6_bundle["operating_threshold"]); v7_probability_map["Version 6 Phase 2 Random Forest"]=v7_v6_prob
for decision,cut in [("V6 frozen threshold",v7_v6_cut),("fixed 0.50",.5)]: v7_final_results.append({"Model":"Version 6 Phase 2 Random Forest","Decision":decision,**v7_metrics(v7_test_y,v7_v6_prob,cut)})
v7_test_metrics=pd.DataFrame(v7_final_results).set_index(["Model","Decision"]).sort_index()
v7_test_metrics.to_csv(V7_ROOT/"descriptive_test_comparison_2020_2024.csv")
print("Post-hoc descriptive comparison on Version 6's previously examined 2020-2024 period; no choices changed after labels opened.")
print("V6 artifact:",v7_v6_artifact_used,"| rows:",len(v7_test_y),"| positive rows:",int(v7_test_y.sum()),"| prevalence:",round(float(v7_test_y.mean()),4)); display(v7_test_metrics.round(4))
# 2018-2019 has no positive labels in the preserved dataset; treat it as false-alarm diagnostics only.
v7_v6_neg_X=goesh.loc[v7_x.index[v7_masks["negative_diagnostic_2018_2019"]],v7_v6_cols]
v7_v6_neg_raw=v7_v6_bundle["model"].predict_proba(v7_v6_neg_X)[:,1]
v7_v6_neg_prob=v7_v6_bundle["probability_calibrator"].predict_proba(v7_logit(v7_v6_neg_raw))[:,1]
v7_negative_probability_map={**v7_negative_probabilities,"Version 6 Phase 2 Random Forest":v7_v6_neg_prob}; v7_negative_rows=[]
for name,prob in v7_negative_probability_map.items():
    cut=v7_v6_cut if name=="Version 6 Phase 2 Random Forest" else v7_thresholds[name]["operating"]
    pred=np.asarray(prob>=cut,dtype="int8"); only_negative=int(v7_negative_y.sum())==0
    v7_negative_rows.append({"Period":"2018-2019","Model":name,"Threshold":float(cut),"Rows":len(v7_negative_y),"Positive_labels":int(v7_negative_y.sum()),"False_alarms":int(pred.sum()),"FPR":float(pred.mean()) if only_negative and len(pred) else np.nan,"Alarm_rate":float(pred.mean()) if len(pred) else np.nan,"Interpretation":"negative-only; recall/AUC not estimable" if only_negative else "mixed labels"})
v7_negative_diagnostics=pd.DataFrame(v7_negative_rows); v7_negative_diagnostics.to_csv(V7_ROOT/"negative_only_false_alarm_diagnostic_2018_2019.csv",index=False)
print("2018-2019 diagnostic | rows:",len(v7_negative_y),"positive labels:",int(v7_negative_y.sum())); display(v7_negative_diagnostics.round(4))
# Annual and broad calendar-phase summaries are descriptive, not independent validation.
v7_annual_rows=[]
for year in range(2020,2025):
    mask=v7_test_y.index.year==year
    if not mask.any():
        print("No matched evaluation rows for year",year,"; skipped annual metrics.")
        continue
    for name in [v7_champion_family,"Version 6 Phase 2 Random Forest"]:
        prob=v7_probability_map[name][mask]; cut=v7_thresholds[v7_champion_family]["operating"] if name==v7_champion_family else v7_v6_cut
        v7_annual_rows.append({"Year":year,"Model":name,**v7_metrics(v7_test_y.loc[mask],prob,cut)})
v7_annual_metrics=pd.DataFrame(v7_annual_rows); v7_annual_metrics.to_csv(V7_ROOT/"descriptive_annual_metrics_2020_2024.csv",index=False); display(v7_annual_metrics.round(4))
v7_cycle_blocks={"2020-2021 early rise":(2020,2021),"2022-2023 rising phase":(2022,2023),"2024 near-maximum phase":(2024,2024)}; v7_cycle_rows=[]
for period,(first,last) in v7_cycle_blocks.items():
    mask=v7_test_y.index.year.isin(range(first,last+1))
    if not mask.any():
        print("No matched evaluation rows for calendar phase",period,"; skipped.")
        continue
    for name in [v7_champion_family,"Version 6 Phase 2 Random Forest"]:
        prob=v7_probability_map[name][mask]; cut=v7_thresholds[v7_champion_family]["operating"] if name==v7_champion_family else v7_v6_cut
        v7_cycle_rows.append({"Calendar_phase_proxy":period,"Model":name,**v7_metrics(v7_test_y.loc[mask],prob,cut)})
v7_cycle_metrics=pd.DataFrame(v7_cycle_rows); v7_cycle_metrics.to_csv(V7_ROOT/"descriptive_calendar_phase_metrics_2020_2024.csv",index=False); display(v7_cycle_metrics.round(4))


In [ ]:
# Ablations, charts, deployment bundle, and reload/prediction verification.
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import roc_curve,precision_recall_curve,confusion_matrix
from sklearn.calibration import calibration_curve
v7_ablation_rows=[]
for name,prob in v7_ablation_test_probabilities.items():
    cut=v7_ablation_thresholds[name]["operating"]
    v7_ablation_rows.append({"Ablation":name,**v7_metrics(v7_test_y,prob,cut)})
v7_ablation_metrics=pd.DataFrame(v7_ablation_rows).set_index("Ablation").sort_index()
v7_ablation_metrics.to_csv(V7_ROOT/"ablation_descriptive_test_2020_2024.csv")
print("Post-hoc descriptive feature ablations; models and thresholds were frozen before 2018:"); display(v7_ablation_metrics.round(4))
v7_ablation_negative_rows=[]
for name,prob in v7_ablation_negative_probabilities.items():
    pred=np.asarray(prob>=v7_ablation_thresholds[name]["operating"],dtype="int8")
    v7_ablation_negative_rows.append({"Ablation":name,"Threshold":v7_ablation_thresholds[name]["operating"],"Rows":len(v7_negative_y),"Positive_labels":int(v7_negative_y.sum()),"False_alarms":int(pred.sum()),"Alarm_rate":float(pred.mean()) if len(pred) else np.nan})
pd.DataFrame(v7_ablation_negative_rows).to_csv(V7_ROOT/"ablation_negative_diagnostic_2018_2019.csv",index=False)

v7_fig,v7_axes=plt.subplots(1,3,figsize=(19,5.5))
for name in [v7_champion_family,"Soft-voting ensemble","Version 6 Phase 2 Random Forest"]:
    prob=v7_probability_map[name]; fpr,tpr,_=roc_curve(v7_test_y,prob); prec,rec,_=precision_recall_curve(v7_test_y,prob)
    frac,mean=calibration_curve(v7_test_y,prob,n_bins=10,strategy="quantile")
    v7_axes[0].plot(fpr,tpr,label=f"{name} AUC={roc_auc_score(v7_test_y,prob):.3f}")
    v7_axes[1].plot(rec,prec,label=f"{name} AP={average_precision_score(v7_test_y,prob):.3f}")
    v7_axes[2].plot(mean,frac,marker=".",label=name)
v7_axes[0].plot([0,1],[0,1],"k--",lw=1); v7_axes[0].set(title="Previously examined period ROC (2020-2024)",xlabel="False positive rate",ylabel="Recall")
v7_axes[1].set(title="2020-2024 precision-recall",xlabel="Recall",ylabel="Precision")
v7_axes[2].plot([0,1],[0,1],"k--",lw=1,label="Ideal"); v7_axes[2].set(title="2020-2024 calibration",xlabel="Mean predicted probability",ylabel="Observed event rate")
for ax in v7_axes: ax.grid(alpha=.25); ax.legend(fontsize=7)
v7_fig.tight_layout(); v7_fig.savefig(V7_ROOT/"descriptive_test_performance_2020_2024.png",dpi=170,bbox_inches="tight"); plt.show()
v7_cm=confusion_matrix(v7_test_y,v7_test_probabilities[v7_champion_family]>=v7_thresholds[v7_champion_family]["operating"],labels=[0,1])
fig,ax=plt.subplots(figsize=(5.5,4.5)); ax.imshow(v7_cm,cmap="Blues")
for (i,j),value in np.ndenumerate(v7_cm): ax.text(j,i,f"{value:,}",ha="center",va="center")
ax.set(xticks=[0,1],yticks=[0,1],xticklabels=["Below M","M/X"],yticklabels=["Below M","M/X"],xlabel="Predicted",ylabel="Observed",title=f"{v7_champion_family}: frozen 2017 threshold")
fig.tight_layout(); fig.savefig(V7_ROOT/"champion_confusion_2020_2024.png",dpi=170,bbox_inches="tight"); plt.show()

v7_threshold_payload={"threshold_objective":"0.40*Recall + 0.40*TSS + 0.20*Precision","threshold_validation_period":"2017; 24-hour purge","feature_periods":{"ranking":"2010-2012","subset_validation":"2013","Optuna_CV":[2014,2015]},"calibration_period":"2016","champion":v7_champion_thresholds,"ensemble":v7_thresholds["Soft-voting ensemble"],"all_families":v7_thresholds,"ablation_thresholds":v7_ablation_thresholds}
(V7_ROOT/"risk_thresholds.json").write_text(json.dumps(v7_threshold_payload,indent=2,default=str))
v7_feature_schema={"version":"Solaris AI Version 8","target":"SuryaBench label_max: M/X flare within next 24 hours","timezone":"UTC","horizon_hours":24,"selected_features":list(v7_selected_features),"feature_groups":{k:list(v) for k,v in v7_groups.items()},"goes_features":list(feature_cols),"sharp_numeric_keywords":list(v7_sharp_metrics),"sunspot_features":[c for c in v7_all_features if c.startswith("sunspot_")],"flare_history_features":[c for c in v7_all_features if c.startswith("flare_")],"availability_delay_hours":{"GOES":1,"SHARP":V7_SHARP_DELAY_HOURS,"SRS":24,"flare_event_after_peak":V7_EVENT_DELAY_HOURS},"selection":"Spearman/MI on 2010-2012; permutation fit through 2011 and scored in 2012; SHAP on training; feature count using fold-local rankings in 2012 and 2013.","validation":"Optuna 2014-2015; final fit through 2015; calibration 2016; thresholds 2017; horizon and observed active-region purges.","limitations":["2018-2019 is negative-only diagnostic.","2020-2024 was previously examined by Version 6; comparison is descriptive, not independent.","Retrospective flare-catalog latency is approximated, not live-validated."],"missingness":"Unavailable source values remain missing; fitted pipeline imputes medians and adds missing indicators."}
(V7_ROOT/"feature_schema.json").write_text(json.dumps(v7_feature_schema,indent=2))
v7_bundle={"version":"Solaris AI Version 8","model_family":v7_champion_family,"model":v7_champion_model,"preprocessing_pipeline":v7_champion_model,"probability_calibrator":v7_champion_calibrator,"feature_columns":list(v7_selected_features),"feature_schema":v7_feature_schema,"risk_thresholds":v7_champion_thresholds,"calibration_period":"2016","threshold_validation_period":"2017","descriptive_period":"2020-2024, previously examined by Version 6; not independent","negative_diagnostic_period":"2018-2019","v6_phase2_reference":v7_v6_artifact_used,"availability_delays_hours":{"GOES":1,"SHARP":V7_SHARP_DELAY_HOURS,"SRS":24,"flare_event_after_peak":V7_EVENT_DELAY_HOURS}}
v7_model_path=V7_ROOT/"solaris_ai_v8_best_model.joblib"; joblib.dump(v7_bundle,v7_model_path,compress=3)
joblib.dump(v7_champion_model,V7_ROOT/"preprocessing_and_model_pipeline.joblib",compress=3)
joblib.dump(v7_champion_calibrator,V7_ROOT/"probability_calibrator.joblib",compress=3)
(V7_ROOT/"selected_features.json").write_text(json.dumps(list(v7_selected_features),indent=2))
joblib.dump({"models":v7_final_models,"calibrators":v7_calibrators,"thresholds":v7_thresholds,"ablation_models":v7_ablation_models,"ablation_calibrators":v7_ablation_calibrators,"ablation_thresholds":v7_ablation_thresholds},V7_ROOT/"solaris_ai_v8_model_comparison.joblib",compress=3)
v7_reloaded=joblib.load(v7_model_path); v7_reload_X=v7_test_X.loc[:,v7_reloaded["feature_columns"]]
v7_reload_raw=v7_reloaded["model"].predict_proba(v7_reload_X)[:,1]
v7_reload_prob=v7_reloaded["probability_calibrator"].predict_proba(v7_logit(v7_reload_raw))[:,1]
assert np.allclose(v7_reload_prob,v7_test_probabilities[v7_champion_family],atol=1e-10,rtol=1e-8)
print("Model reload verified; max probability delta:",float(np.max(np.abs(v7_reload_prob-v7_test_probabilities[v7_champion_family]))))
v7_inference_py="""import joblib
import numpy as np
import pandas as pd
def predict_solaris_v8(model_path, engineered_features):
    bundle=joblib.load(model_path); columns=bundle["feature_columns"]
    missing=sorted(set(columns)-set(engineered_features.columns))
    if missing: raise ValueError(f"Missing Version 8 feature columns: {missing}")
    X=engineered_features.loc[:,columns].replace([np.inf,-np.inf],np.nan)
    raw=np.clip(bundle["model"].predict_proba(X)[:,1],1e-6,1-1e-6)
    z=np.log(raw/(1-raw)).reshape(-1,1)
    probability=bundle["probability_calibrator"].predict_proba(z)[:,1]; cuts=bundle["risk_thresholds"]
    return pd.DataFrame({"probability_mx_next_24h":probability,"watch_threshold_recall_90":probability>=cuts["watch_recall_90"],"operating_alert":probability>=cuts["operating"],"high_precision_alert":(probability>=cuts["high_precision_70"]) if cuts["high_precision_70"] is not None else False},index=engineered_features.index)
"""
(V7_ROOT/"deployment_inference.py").write_text(v7_inference_py)
v7_champion_metrics=v7_metrics(v7_test_y,v7_test_probabilities[v7_champion_family],v7_thresholds[v7_champion_family]["operating"])
v7_v6_metrics=v7_metrics(v7_test_y,v7_v6_prob,v7_v6_cut)
v7_improvement={k:float(v7_champion_metrics[k]-v7_v6_metrics[k]) for k in ["Accuracy","Precision","Recall","F1","TSS","ROC_AUC","PR_AUC","Brier","Log_loss","ECE_10_bins","FPR"]}
v7_report={"best_model_selected_before_evaluation":v7_champion_family,"best_parameters":v7_best_params[v7_champion_family],"selected_feature_count":len(v7_selected_features),"descriptive_evaluation_period":"2020-2024, previously examined by Version 6; not independent","descriptive_rows":int(len(v7_test_y)),"descriptive_positive_rate":float(v7_test_y.mean()),"champion_metrics":v7_champion_metrics,"v6_phase2_metrics":v7_v6_metrics,"improvement_v7_minus_v6":v7_improvement,"descriptive_accuracy_at_least_95_percent":bool(v7_champion_metrics["Accuracy"]>=.95),"negative_2018_2019":{"rows":len(v7_negative_y),"positive_labels":int(v7_negative_y.sum()),"metrics_file":"negative_only_false_alarm_diagnostic_2018_2019.csv"},"limitations":V7_SOURCE_LIMITATIONS,"active_region_purges":V7_ACTIVE_REGION_PURGES,"sources":{"SHARP":{"verified_historical_coverage":bool(v7_sharp_verified),"rows":len(v7_sharp),"years_present":v7_years_present if len(v7_sharp) else [],"finite_metric_counts":v7_metric_counts if len(v7_sharp) else {},"annual_coverage_file":"sharp_annual_coverage.csv","yearly_failures":v7_sharp_failures,"numeric_only":True},"SRS":{"available":bool(v7_srs_available),"coverage":float(v7_srs_coverage),"parsed_coverage":float(v7_srs_parse_coverage),"parsed_reports":len(v7_srs),"request_errors":len(v7_srs_errors),"annual_coverage_file":"noaa_srs_annual_coverage.csv"},"GOES_MX_flare_events":len(v7_flare)},"metrics_file":"descriptive_test_comparison_2020_2024.csv","model_file":str(v7_model_path),"choice_policy":"No choices changed after evaluation labels were opened."}
(V7_ROOT/"run_report.json").write_text(json.dumps(v7_report,indent=2,default=str))
(V7_ROOT/"README.txt").write_text("Solaris AI Version 8 artifacts\n"+f"Champion: {v7_champion_family}\nDescriptive 2020-2024 accuracy: {v7_champion_metrics['Accuracy']:.4f}\nV6 Phase 2 accuracy at frozen threshold: {v7_v6_metrics['Accuracy']:.4f}\nAccuracy delta: {v7_improvement['Accuracy']:+.4f}\nDescriptive accuracy at least 95% (not independent): {v7_champion_metrics['Accuracy']>=.95}\n2020-2024 was previously examined; this comparison is not independent.\n2018-2019 has no positive labels and is used only for false-alarm diagnostics.\n")
print("Saved model:",v7_model_path,"| champion:",v7_champion_family,"| accuracy:",round(v7_champion_metrics["Accuracy"],4),"| V6 delta:",round(v7_improvement["Accuracy"],4))
print("Descriptive accuracy at least 95% (not independent):",v7_champion_metrics["Accuracy"]>=.95)
